# Enhanced Game v2 — a phase-aware, throughput-benchmarked Kaggriculture league

Version 2 builds directly on Enhanced Game v1 and keeps the same bounded, resumable
self-play league. Three measured changes:

1. **Phase-dependent policy.** The policy can now use different `crop_bias`,
   `plant_floor` and `deposit_bias` in the late game (from `late_day` onward). With
   no `*_late` keys a config reproduces v1 exactly; the search decides whether a
   distinct late-game policy actually wins.
2. **Throughput auto-benchmark.** Before training, the notebook times games at a
   few worker counts and picks the fastest, instead of the fixed `min(2, cpus)`.
   Matches are CPU Python processes; this is the real throughput lever (a GPU does
   not accelerate this simulator — see *Method and limits*).
3. **One widened bound.** `plant_floor` gains an 80 option, the only parameter with
   evidence of pressing against its ceiling in the V12 champion.

Everything else is unchanged: promotion still requires beating V9, the champion and
every opponent on fresh holdout seeds, with positive bootstrap margins. If no
challenger qualifies, the V12 baseline is retained. The notebook exports `main.py`
and a checkpoint; it makes no Kaggle submission.


## What counts as progress?

A challenger is promoted only if, on fresh unseen seeds, it beats **V9** (>=80%),
the **current champion** (>=60%) and **every** league opponent (>=55%) with positive
mean margins and positive seed-bootstrap lower bounds. These gates are unchanged
from v1.

Measured context this version was built on (single-CPU sandbox, small samples, not a
leaderboard): V12 trial-23's edge over V9 head-to-head was small and noisy (about a
coin flip on 12 games), and V12 chose a compact two-quadrant farm rather than
expanding to all four. So a real improvement means beating V9 *decisively and
repeatably*, which only a full validated run can demonstrate. No leaderboard gain is
promised here.


In [ ]:
from pathlib import Path
import os

TIME_BUDGET_MINUTES = 480       # Per session; adjust to your runtime allowance
MAX_CYCLES = 30                # Total across resumed sessions
GENERAL_TRIALS = 14            # Per cycle (raised slightly: v2 search space is larger)
EXPLOITER_TRIALS = 6
WORKERS = min(2, os.cpu_count() or 2)   # Safe fallback; the benchmark cell may raise this
AUTO_BENCHMARK_WORKERS = True  # Time a few worker counts and pick the fastest
BENCH_GAMES = 8                # Games per candidate worker count in the benchmark
TRAIN_SEEDS = 3
SELECTION_SEEDS = 4
HOLDOUT_SEEDS = 24             # Used for each holdout / confirmation / counter test
FINALISTS = 2                  # Top generalists plus one exploiter
SELFPLAY_SEEDS = 2
HISTORY_SIZE = 4               # Active history; all old files remain in the archive

TARGET_MEAN_CASH = 100_000
TARGET_WIN_RATE = .75
TARGET_80K_RATE = .50
STOP_ON_TARGET = True
STAGNATION_CYCLES = 6          # Increase on resume to continue a stalled search
V9_WIN_RATE = .80
CHAMPION_WIN_RATE = .60
MIN_OPPONENT_WIN_RATE = .55
MAX_DECISION_SECONDS = .80

RESUME_ZIP = ''                # Only Enhanced Game v2 checkpoints (v1/V12/V13 are not compatible)
EXTRA_OPPONENTS = []           # Trusted executable .py policies, NOT replay JSONs
MOUNT_GOOGLE_DRIVE = False
CHECKPOINT_COPY = ''           # e.g. /content/drive/MyDrive/enhanced_game_v2_checkpoint.zip

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else (Path('/content') if Path('/content').exists() else Path.cwd())
OUTPUT = WORKING / 'enhanced_game_v2'
CODE = WORKING / 'enhanced_game_v2_code'
OUTPUT.mkdir(parents=True,exist_ok=True)
CODE.mkdir(parents=True,exist_ok=True)
assert TIME_BUDGET_MINUTES>0 and 1<=WORKERS<=(os.cpu_count() or 2)
assert 1<=MAX_CYCLES<10000 and STAGNATION_CYCLES>=1 and HISTORY_SIZE>=0
assert GENERAL_TRIALS>=4 and EXPLOITER_TRIALS>=2 and FINALISTS>=1
assert 1<=TRAIN_SEEDS<10000 and 1<=SELECTION_SEEDS<10000 and 1<=SELFPLAY_SEEDS<10000
assert 24<=HOLDOUT_SEEDS<10000
assert .8<=V9_WIN_RATE<=1 and .6<=CHAMPION_WIN_RATE<=1 and .55<=MIN_OPPONENT_WIN_RATE<=1
assert 0<TARGET_WIN_RATE<=1 and 0<=TARGET_80K_RATE<=1 and TARGET_MEAN_CASH>0
print('Output directory:',OUTPUT)

## Setup

The simulator is pinned to 1.32.7, as in the preceding notebooks. Verify the competition engine version before deliberately changing it. The installer avoids `ensurepip`, which previously failed on Kaggle. GPU/TPU selection does not accelerate this parameter-search implementation.


In [ ]:
import subprocess, sys, venv
ENV = WORKING / '.enhanced_game_v2_env'
PYTHON = ENV / 'bin' / 'python'
venv.EnvBuilder(with_pip=False, system_site_packages=True).create(ENV)
pip=[sys.executable,'-m','pip','--python',str(PYTHON)]
subprocess.run(pip+['install','--quiet','--no-deps','kaggle-environments==1.32.7'],check=True)
subprocess.run(pip+['install','--quiet','optuna==4.5.0','jsonschema','requests','numpy'],check=True)
subprocess.run([str(PYTHON),'-c',"from kaggle_environments import make; make('kaggriculture', configuration={'seed': 1}); print('Simulator ready')"],check=True)

## Embedded agents and learning loop

No previous uploads are needed for a fresh run. The V12 fallback and V9/V11 anchors are included exactly. Training changes the parameterized Enhanced Game policy, not those baseline files. V13 runtime helpers are reused, but the league controller below owns training, selection and promotion.

Learning happens between games on the notebook. The uploaded `main.py` responds to observations using its learned settings; it does not run the training league during a Kaggle match. Only legally visible opponent farm information is used. Hidden opponent inventories are never read by the policy.


In [ ]:
EMBEDDED_FILES = {'league.py': '"""Enhanced Game v2: bounded, resumable league self-play with phase-dependent policy search.\n\nTPE fits a model over explicit policy parameters. No external account access.\n"""\nimport argparse\nimport concurrent.futures\nimport contextlib\nimport io\nimport json\nimport math\nimport multiprocessing\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport sqlite3\nimport statistics\nimport time\nimport zipfile\nimport optuna\nimport support\n\nBASE=dict(support.BASE)\nSPACE={\n    \'animals\':(\'int\',6,22,2), \'hands\':(\'int\',8,12,1), \'land\':(\'int\',2,4,1),\n    \'crop_bias\':(\'float\',.6,2.5), \'care_bias\':(\'float\',.6,2.),\n    \'fert_bias\':(\'float\',.6,1.8), \'opponent_weight\':(\'cat\',0.,.5,1.),\n    \'drop_units\':(\'int\',2,6,1), \'drop_value\':(\'cat\',250,500,1000,1000000),\n    \'deposit_bias\':(\'float\',.15,.65), \'care_cap\':(\'cat\',1.3,2.,3.),\n    \'plant_floor\':(\'cat\',12,25,40,60,80), \'hire_pace\':(\'int\',1,3,1),\n    \'workload_hiring\':(\'cat\',False,True), \'work_per_hand\':(\'cat\',6,8,10,12),\n    \'land_util\':(\'cat\',0.,.35,.45,.55,.75), \'land_buffer\':(\'cat\',300,700,1500),\n    \'commitment\':(\'cat\',1.,1.3,1.7), \'region_weight\':(\'cat\',.65,.8,1.),\n    \'distance_weight\':(\'cat\',.45,.65,.85), \'dig_value\':(\'cat\',20,45,75),\n    \'animal_deadline\':(\'cat\',12,16,19), \'expansion_hands\':(\'cat\',7,9,11),\n    \'night_deposit\':(\'cat\',False,True),\n    # Alternative strategies for exploiters, beyond a larger farm.\n    \'crop_style\':(\'cat\',\'balanced\',\'quick\',\'orchard\'),\n    \'animal_style\':(\'cat\',\'balanced\',\'milk\',\'wool\',\'eggs\'),\n    # Enhanced Game v2: late-game phase overrides (day>=late_day). Absent/default\n    # values reproduce v1; the search explores whether a distinct late policy wins.\n    \'late_day\':(\'cat\',20,24,27),\n    \'crop_bias_late\':(\'float\',.6,2.5),\n    \'plant_floor_late\':(\'cat\',12,25,40,60,80),\n    \'deposit_bias_late\':(\'float\',.15,.65),\n}\n\ndef sample(trial):\n    p=dict(BASE)\n    for k,s in SPACE.items():\n        if s[0]==\'int\':p[k]=trial.suggest_int(k,s[1],s[2],step=s[3])\n        elif s[0]==\'float\':p[k]=trial.suggest_float(k,s[1],s[2])\n        else:p[k]=trial.suggest_categorical(k,list(s[1:]))\n    return p\n\ndef warm_params(p):\n    p=dict(BASE,**p)\n    p.setdefault(\'crop_style\',\'balanced\');p.setdefault(\'animal_style\',\'balanced\')\n    for k,s in SPACE.items():\n        if s[0]==\'cat\' and p[k] not in s[1:]:p[k]=s[1]\n        elif s[0] in (\'int\',\'float\'):\n            p[k]=max(s[1],min(s[2],p[k]))\n            if s[0]==\'int\':p[k]=s[1]+round((p[k]-s[1])/s[3])*s[3]\n    return {k:p[k] for k in SPACE}\n\ndef read_cfg(path):\n    import ast\n    for node in ast.parse(Path(path).read_text()).body:\n        if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'CFG\' for t in node.targets):\n            return ast.literal_eval(node.value)\n    return {}\n\ndef evaluate_gate(rows,cfg,champion_name):\n    """No promotion from training or too few fresh seed clusters."""\n    groups={name:[r for r in rows if r[\'opponent\']==name] for name in sorted({r[\'opponent\'] for r in rows})}\n    per={n:support.summary(g) for n,g in groups.items()}\n    intervals={n:support.seed_margin_interval(g) for n,g in groups.items()}\n    required={\'main_v9.py\',champion_name}\n    enough=bool(groups) and all(len({r[\'seed\'] for r in g})>=24 for g in groups.values())\n    normal=all(r[\'normal\'] and r[\'max_decision_seconds\']<cfg[\'max_decision_seconds\'] for r in rows)\n    baseline_ok=required.issubset(groups)\n    if baseline_ok:\n        baseline_ok=(per[\'main_v9.py\'][\'win_rate\']>=cfg[\'v9_win_rate\'] and intervals[\'main_v9.py\'][0]>0 and\n                     per[champion_name][\'win_rate\']>=cfg[\'champion_win_rate\'] and intervals[champion_name][0]>0)\n    broad=all(p[\'mean_margin\']>0 and p[\'win_rate\']>=cfg[\'min_opponent_win_rate\'] for p in per.values())\n    pooled=support.seed_margin_interval(rows)\n    promoted=bool(enough and normal and baseline_ok and broad and pooled[0]>0)\n    # Counterstrategy: it demonstrably attacks the current champion, even when\n    # its broader record is insufficient. It joins training, not the export.\n    exploiter=bool(enough and normal and champion_name in per and\n                  per[champion_name][\'win_rate\']>=cfg[\'champion_win_rate\'] and intervals[champion_name][0]>0)\n    return dict(promoted=promoted,exploiter=exploiter,per_opponent=per,\n                margin_intervals=intervals,pooled_margin_interval=pooled,\n                enough_fresh_seeds=enough,normal_and_fast=normal)\n\ndef target_met(rows,cfg):\n    s=support.summary(rows)\n    return (s[\'mean_cash\']>=cfg[\'target_mean_cash\'] and s[\'win_rate\']>=cfg[\'target_win_rate\']\n            and s[\'reach_80k_rate\']>=cfg[\'target_80k_rate\'])\n\nclass League:\n    def __init__(self,cfg,pool):\n        self.cfg=cfg;self.root=Path(cfg[\'output_dir\']).resolve();self.pool=pool\n        self.source=Path(__file__).resolve().parent\n        self.root.mkdir(parents=True,exist_ok=True)\n        for d in [\'agents\',\'candidates\',\'reports\',\'replays\',\'sources\']:(self.root/d).mkdir(exist_ok=True)\n        self.started=time.monotonic();self.deadline=self.started+cfg[\'time_budget_minutes\']*60\n        self.template=(self.source/\'policy_template.py\').read_text()\n        fixed=[]\n        for name in [\'main_v9.py\',\'main_v11.py\',\'main_v12.py\']:\n            shutil.copyfile(self.source/name,self.root/\'agents\'/name);fixed.append(\'agents/\'+name)\n        for name,p in {\n            \'stress_crops.py\':dict(BASE,animals=6,hands=12,land=4,crop_bias=2.,plant_floor=60,land_util=.35,crop_style=\'orchard\'),\n            \'stress_livestock.py\':dict(BASE,animals=20,hands=12,care_bias=1.8,animal_style=\'milk\')}.items():\n            (self.root/\'agents\'/name).write_text(support.render(self.template,p));fixed.append(\'agents/\'+name)\n        for p in cfg.get(\'extra_opponents\',[]):\n            p=Path(p);name=\'external_\'+support.digest(p)[:16]+\'.py\'\n            shutil.copyfile(p,self.root/\'agents\'/name);fixed.append(\'agents/\'+name)\n        self.fixed=fixed\n        contract={k:cfg[k] for k in [\'train_seeds\',\'selection_seeds\',\'holdout_seeds\',\'general_trials\',\'exploiter_trials\',\n                  \'finalists\',\'history_size\',\'selfplay_seeds\',\'v9_win_rate\',\'champion_win_rate\',\'min_opponent_win_rate\',\n                  \'max_decision_seconds\',\'target_mean_cash\',\'target_win_rate\',\'target_80k_rate\']}\n        contract[\'sources\']={n:support.digest(self.source/n) for n in [\'league.py\',\'support.py\',\'policy_template.py\']}\n        contract[\'fixed\']={p:support.digest(self.root/p) for p in fixed}\n        contract[\'environment\']=support.ENV_VERSION\n        manifest=self.root/\'manifest.json\'\n        if manifest.exists() and json.loads(manifest.read_text())!=contract:\n            raise ValueError(\'This checkpoint has different code, opponents or study settings. Start a new output folder.\')\n        support.write_json(manifest,contract)\n        for n in [\'league.py\',\'support.py\',\'policy_template.py\',\'main_v9.py\',\'main_v11.py\',\'main_v12.py\']:\n            shutil.copyfile(self.source/n,self.root/\'sources\'/n)\n        if (self.root/\'state.json\').exists():self.state=json.loads((self.root/\'state.json\').read_text())\n        else:\n            self.state=dict(cycle=0,phase=\'new_cycle\',champion=\'agents/main_v12.py\',\n                            champion_sha=support.digest(self.root/\'agents/main_v12.py\'),\n                            archives=[],tested_hashes=[],no_progress=0,promotions=0,\n                            target_reached=False,restarts=0)\n        for a in self.state[\'archives\']:\n            if support.digest(self.root/a[\'path\'])!=a[\'sha256\']:raise ValueError(\'Archived agent hash mismatch\')\n        if support.digest(self.root/self.state[\'champion\'])!=self.state[\'champion_sha\']:\n            raise ValueError(\'Champion hash mismatch\')\n        self.state[\'restarts\']+=1\n        shutil.copyfile(self.root/self.state[\'champion\'],self.root/\'main.py\')\n        # Reuse the verified game runner; league transitions live only here.\n        rcfg={\'time_budget_minutes\':cfg[\'time_budget_minutes\'],\'workers\':cfg[\'workers\']}\n        self.runner=support.Runner(self.root,rcfg,pool);self.runner.deadline=self.deadline\n        self.save(\'ready\')\n\n    def save(self,status):\n        # A provisional file may have been copied for the loading test when an\n        # interrupt arrived. Checkpoints must only export the accepted champion.\n        accepted=self.root/self.state[\'champion\']\n        exported=self.root/\'main.py\'\n        if not exported.exists() or support.digest(exported)!=self.state[\'champion_sha\']:\n            shutil.copyfile(accepted,exported)\n        support.write_json(self.root/\'state.json\',self.state)\n        decision=dict(status=status,export=self.state[\'champion\'],sha256=self.state[\'champion_sha\'],\n                      promoted=self.state[\'promotions\']>0,promotions=self.state[\'promotions\'],\n                      cycle=self.state[\'cycle\'],phase=self.state[\'phase\'],target_reached=self.state[\'target_reached\'],\n                      latest_report=self.state.get(\'latest_report\'),\n                      note=\'Local league results; no leaderboard rank guarantee. No Kaggle submission was made.\')\n        support.write_json(self.root/\'decision.json\',decision)\n        db=self.root/\'study.db\'\n        if db.exists():\n            with sqlite3.connect(db) as src,sqlite3.connect(self.root/\'study-backup.db\') as dst:src.backup(dst)\n        tmp=self.root/\'checkpoint.zip.tmp\'\n        with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED) as z:\n            for name in [\'state.json\',\'manifest.json\',\'main.py\',\'games.jsonl\',\'decision.json\']:\n                if (self.root/name).exists():z.write(self.root/name,name)\n            if (self.root/\'study-backup.db\').exists():z.write(self.root/\'study-backup.db\',\'study.db\')\n            for folder in [\'agents\',\'candidates\',\'reports\',\'replays\',\'sources\']:\n                for p in (self.root/folder).glob(\'*\'):\n                    if p.is_file():z.write(p,str(p.relative_to(self.root)))\n        tmp.replace(self.root/\'checkpoint.zip\')\n        mirror=self.cfg.get(\'checkpoint_copy\',\'\')\n        if mirror:\n            p=Path(mirror).expanduser()\n            if p.resolve()!=(self.root/\'checkpoint.zip\').resolve():\n                try:\n                    p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+\'.tmp\')\n                    shutil.copyfile(self.root/\'checkpoint.zip\',t);t.replace(p)\n                except OSError as e:print(\'Local checkpoint saved; external copy failed:\',str(e),flush=True)\n\n    def paths(self,names):return [self.root/n for n in names]\n\n    def seeds(self,phase,n):\n        # Disjoint blocks across phases AND cycles. No recycled holdout seeds.\n        return range(30000000+self.state[\'cycle\']*100000+phase*10000,\n                     30000000+self.state[\'cycle\']*100000+phase*10000+n)\n\n    def freeze_league(self):\n        hist=self.state[\'archives\'];limit=self.cfg[\'history_size\']\n        # Keep early history as well as recent counterstrategies. The full archive\n        # is never deleted; this bounds the active evaluation cost.\n        chosen=([hist[0]] if hist and limit else [])\n        if limit>1:chosen+=hist[-(limit-1):]\n        names=self.fixed+[self.state[\'champion\']]+[a[\'path\'] for a in chosen]\n        seen=set();frozen=[]\n        for n in names:\n            sha=support.digest(self.root/n)\n            if sha not in seen:frozen.append(n);seen.add(sha)\n        self.state[\'league\']=frozen\n        self.state[\'frozen_champion\']=next(n for n in frozen if support.digest(self.root/n)==self.state[\'champion_sha\'])\n        self.state[\'phase\']=\'selfplay\'\n        self.save(\'cycle_started\')\n        print(\'CYCLE\',self.state[\'cycle\'],\'champion\',self.state[\'champion\'],\n              \'opponents\',len(frozen),\'archives\',len(hist),flush=True)\n\n    def study(self,role):\n        optuna.logging.set_verbosity(optuna.logging.WARNING)\n        return optuna.create_study(study_name=f"egv1_c{self.state[\'cycle\']:03d}_{role}",\n            direction=\'maximize\',storage=\'sqlite:///\'+str(self.root/\'study.db\'),load_if_exists=True,\n            sampler=optuna.samplers.TPESampler(seed=731+self.state[\'cycle\']*31+self.state[\'restarts\'],n_startup_trials=4))\n\n    def train(self,role):\n        study=self.study(role);total=self.cfg[role+\'_trials\']\n        opponents=([self.state[\'frozen_champion\']] if role==\'exploiter\' else self.state[\'league\'])\n        phase=1 if role==\'general\' else 2\n        if not study.trials:\n            seeds=[warm_params(read_cfg(self.root/self.state[\'frozen_champion\'])),warm_params(BASE)]\n            if role==\'exploiter\':\n                seeds[1]=warm_params(dict(BASE,land=4,animals=8,crop_style=\'quick\',night_deposit=True))\n            seeds+=self.state.get(\'warm_starts\',[])[:2]\n            for p in seeds[:total]:study.enqueue_trial(p)\n        while True:\n            if time.monotonic()>=self.deadline:raise TimeoutError()\n            pending=self.state.get(\'pending_trial\')\n            finished=sum(t.state.is_finished() for t in study.trials)\n            if not pending and finished>=total:break\n            if not pending:\n                trial=study.ask();params=sample(trial)\n                name=f"egv1_c{self.state[\'cycle\']:03d}_{role}_t{trial.number:04d}"\n                path=\'candidates/\'+name+\'.py\'\n                (self.root/path).write_text(support.render(self.template,params))\n                pending=dict(role=role,number=trial.number,params=params,path=path)\n                self.state[\'pending_trial\']=pending;self.save(\'trial_reserved\')\n            if pending[\'role\']!=role:raise ValueError(\'Pending trial belongs to another phase\')\n            trial_no=pending[\'number\'];candidate=self.root/pending[\'path\']\n            try:\n                rows=self.runner.evaluate(candidate,self.paths(opponents),self.seeds(phase,self.cfg[\'train_seeds\']))\n                value=support.score(rows)\n                done=next(t for t in study.trials if t.number==trial_no)\n                if not done.state.is_finished():study.tell(trial_no,value)\n                support.write_json(candidate.with_suffix(\'.json\'),dict(parameters=pending[\'params\'],summary=support.summary(rows),games=rows))\n                print(role,\'trial\',trial_no,\'score\',round(value,4),\'cash\',round(support.summary(rows)[\'mean_cash\']),flush=True)\n            except RuntimeError as e:\n                done=next(t for t in study.trials if t.number==trial_no)\n                if not done.state.is_finished():study.tell(trial_no,state=optuna.trial.TrialState.FAIL)\n                support.write_json(candidate.with_suffix(\'.json\'),{\'error\':str(e),\'parameters\':pending[\'params\']})\n                print(\'Invalid trial\',role,trial_no,str(e),flush=True)\n            self.state.pop(\'pending_trial\',None);self.save(\'training\')\n        self.state[\'phase\']=\'exploiter\' if role==\'general\' else \'select\'\n        self.save(\'training_phase_complete\')\n\n    def select(self):\n        candidates=[];seen=set(self.state[\'tested_hashes\']);seen.add(self.state[\'champion_sha\'])\n        warm=[]\n        for role in [\'general\',\'exploiter\']:\n            good=sorted((t for t in self.study(role).trials if t.state==optuna.trial.TrialState.COMPLETE),key=lambda t:t.value,reverse=True)\n            count=0\n            for t in good:\n                p=self.root/\'candidates\'/f"egv1_c{self.state[\'cycle\']:03d}_{role}_t{t.number:04d}.py"\n                if support.digest(p) in seen:continue\n                seen.add(support.digest(p));candidates.append(p);count+=1\n                warm.append(warm_params(read_cfg(p)))\n                if count>=(self.cfg[\'finalists\'] if role==\'general\' else 1):break\n        self.state[\'warm_starts\']=warm[:4]\n        if not candidates:\n            self.finish_cycle(False,\'no_untested_candidate\');return\n        scores=[];attack_scores=[]\n        for p in candidates:\n            print(\'Selection\',p.name,flush=True)\n            rows=self.runner.evaluate(p,self.paths(self.state[\'league\']),self.seeds(3,self.cfg[\'selection_seeds\']))\n            result=dict(candidate=p.name,score=support.score(rows),summary=support.summary(rows),games=rows)\n            support.write_json(self.root/\'reports\'/(\'selection_\'+p.stem+\'.json\'),result)\n            scores.append((result[\'score\'],str(p.relative_to(self.root))))\n            if \'_exploiter_\' in p.name:\n                direct=[r for r in rows if r[\'opponent\']==Path(self.state[\'frozen_champion\']).name]\n                attack_scores.append((support.score(direct),str(p.relative_to(self.root))))\n            self.save(\'selection\')\n        self.state[\'selected\']=max(scores)[1]\n        self.state.pop(\'selected_counter\',None)\n        if attack_scores:\n            attacker=max(attack_scores)[1]\n            if attacker!=self.state[\'selected\']:\n                self.state[\'selected_counter\']=attacker\n                self.state[\'tested_hashes\'].append(support.digest(self.root/attacker))\n        self.state[\'tested_hashes\'].append(support.digest(self.root/self.state[\'selected\']))\n        self.state[\'phase\']=\'holdout\';self.save(\'holdout_reserved\')\n\n    def test(self,confirmation=False):\n        candidate=self.root/self.state[\'selected\'];phase=5 if confirmation else 4\n        rows=self.runner.evaluate(candidate,self.paths(self.state[\'league\']),self.seeds(phase,self.cfg[\'holdout_seeds\']))\n        gate=evaluate_gate(rows,self.cfg,Path(self.state[\'frozen_champion\']).name)\n        # Generate a loss replay from the first matching cached game. Replays are\n        # optional diagnostics, never substituted for an adaptive opponent.\n        if not confirmation and time.monotonic()<self.deadline-30:\n            loss=next((r for r in rows if r[\'cash\']<r[\'rival_cash\']),None)\n            if loss:\n                opponent=next(p for p in self.paths(self.state[\'league\']) if p.name==loss[\'opponent\'])\n                replay=self.root/\'replays\'/f"cycle_{self.state[\'cycle\']:03d}_loss.json.gz"\n                if not replay.exists():support.worker((str(candidate),str(opponent),loss[\'seed\'],loss[\'seat\'],str(replay)))\n        name=f"reports/{\'confirmation\' if confirmation else \'holdout\'}_c{self.state[\'cycle\']:03d}.json"\n        record=dict(candidate=self.state[\'selected\'],sha256=support.digest(candidate),gate=gate,\n                    summary=support.summary(rows),target_metrics_met=target_met(rows,self.cfg),games=rows)\n        support.write_json(self.root/name,record);self.state[\'latest_report\']=name\n        self.state[\'pending_decision\']=dict(report=name,confirmation=confirmation)\n        self.state[\'phase\']=\'commit\';self.save(\'evaluation_complete\')\n\n    def archive(self,path,kind):\n        sha=support.digest(path)\n        for a in self.state[\'archives\']:\n            if a[\'sha256\']==sha:return a[\'path\']\n        target=f"agents/egv1_{kind}_{sha[:16]}.py"\n        shutil.copyfile(path,self.root/target)\n        self.state[\'archives\'].append(dict(path=target,sha256=sha,kind=kind,cycle=self.state[\'cycle\']))\n        return target\n\n    def commit(self):\n        pending=self.state[\'pending_decision\'];report=json.loads((self.root/pending[\'report\']).read_text())\n        gate=report[\'gate\'];candidate=self.root/self.state[\'selected\']\n        if pending[\'confirmation\']:\n            success=gate[\'promoted\'] and report[\'target_metrics_met\']\n            self.state[\'target_reached\']=bool(success)\n            if success:self.finish_cycle(True,\'target_confirmed\')\n            else:self.finish_or_counter(True,\'target_confirmation_failed\')\n            return\n        improved=gate[\'promoted\'];added=False\n        if improved:\n            if time.monotonic()>=self.deadline:raise TimeoutError()\n            shutil.copyfile(candidate,self.root/\'main.py\')\n            try:\n                report[\'file_loading_check\']=support.verify_export(self.root,self.root/\'agents/main_v9.py\',\n                    30000000+self.state[\'cycle\']*100000+60000)\n            except Exception:\n                shutil.copyfile(self.root/self.state[\'champion\'],self.root/\'main.py\');raise\n            support.write_json(self.root/pending[\'report\'],report)\n            # Preserve the old champion to limit forgetting.\n            self.archive(self.root/self.state[\'champion\'],\'history\')\n            path=self.archive(candidate,\'champion\')\n            self.state.update(champion=path,champion_sha=support.digest(candidate),target_reached=False)\n            self.state[\'promotions\']+=1\n            print(\'PROMOTED\',path,flush=True)\n        elif gate[\'exploiter\']:\n            known={a[\'sha256\'] for a in self.state[\'archives\']}\n            added=support.digest(candidate) not in known\n            if added:\n                self.archive(candidate,\'counter\')\n                print(\'New counterstrategy archived; champion retained.\',flush=True)\n        if improved and report[\'target_metrics_met\']:\n            self.state[\'phase\']=\'confirm\';self.save(\'target_requires_confirmation\')\n        else:self.finish_or_counter(improved or added,\'champion_promoted\' if improved else (\'counter_added\' if added else \'candidate_rejected\'))\n\n    def finish_or_counter(self,progress,status):\n        if self.state.get(\'selected_counter\'):\n            self.state[\'cycle_progress\']=progress;self.state[\'cycle_status\']=status\n            self.state[\'phase\']=\'counter\';self.save(\'counter_evaluation_reserved\')\n        else:self.finish_cycle(progress,status)\n\n    def counter(self):\n        candidate=self.root/self.state[\'selected_counter\']\n        opponent=self.root/self.state[\'frozen_champion\']\n        rows=self.runner.evaluate(candidate,[opponent],self.seeds(7,self.cfg[\'holdout_seeds\']))\n        gate=evaluate_gate(rows,self.cfg,opponent.name)\n        name=f"reports/counter_c{self.state[\'cycle\']:03d}.json"\n        support.write_json(self.root/name,dict(candidate=str(candidate.relative_to(self.root)),\n                           summary=support.summary(rows),gate=gate,games=rows))\n        added=False\n        if gate[\'exploiter\']:\n            added=support.digest(candidate) not in {a[\'sha256\'] for a in self.state[\'archives\']}\n            self.archive(candidate,\'counter\')\n        self.finish_cycle(added or self.state.get(\'cycle_progress\',False),\n                          \'counter_added\' if added else self.state.get(\'cycle_status\',\'counter_rejected\'))\n\n    def finish_cycle(self,progress,status):\n        self.state[\'no_progress\']=0 if progress else self.state[\'no_progress\']+1\n        self.state[\'cycle\']+=1;self.state[\'phase\']=\'new_cycle\'\n        self.state.pop(\'pending_decision\',None)\n        self.save(status)\n        print(\'Cycle finished:\',status,\'promotions\',self.state[\'promotions\'],flush=True)\n\n    def run(self):\n        while self.state[\'cycle\']<self.cfg[\'max_cycles\']:\n            if self.state[\'target_reached\'] and self.cfg[\'stop_on_target\']:\n                self.save(\'target_confirmed\');return\n            if self.state[\'no_progress\']>=self.cfg[\'stagnation_cycles\']:\n                self.save(\'stagnation_limit\');return\n            if time.monotonic()>=self.deadline:raise TimeoutError()\n            phase=self.state[\'phase\']\n            if phase==\'new_cycle\':self.freeze_league()\n            elif phase==\'selfplay\':\n                p=self.root/self.state[\'champion\']\n                rows=self.runner.evaluate(p,[p],self.seeds(0,self.cfg[\'selfplay_seeds\']))\n                support.write_json(self.root/\'reports\'/f"selfplay_c{self.state[\'cycle\']:03d}.json",\n                                   dict(summary=support.summary(rows),games=rows,note=\'Self-play does not establish relative strength.\'))\n                self.state[\'phase\']=\'general\';self.save(\'selfplay_complete\')\n            elif phase in (\'general\',\'exploiter\'):self.train(phase)\n            elif phase==\'select\':self.select()\n            elif phase==\'holdout\':self.test()\n            elif phase==\'commit\':self.commit()\n            elif phase==\'confirm\':self.test(confirmation=True)\n            elif phase==\'counter\':self.counter()\n            else:raise ValueError(\'Unknown phase: \'+str(phase))\n        self.save(\'target_confirmed\' if self.state[\'target_reached\'] and self.cfg[\'stop_on_target\'] else \'cycle_limit\')\n\ndef main(config_path):\n    cfg=json.loads(Path(config_path).read_text())\n    for k in [\'train_seeds\',\'selection_seeds\',\'holdout_seeds\',\'selfplay_seeds\']:\n        if not 1<=cfg[k]<10000:raise ValueError(\'Seed counts must be 1..9999\')\n    if cfg[\'max_cycles\']>=10000:raise ValueError(\'Use fewer than 10000 cycles\')\n    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):\n        import kaggle_environments\n    if kaggle_environments.__version__!=support.ENV_VERSION:raise ValueError(\'Simulator version mismatch\')\n    with concurrent.futures.ProcessPoolExecutor(max_workers=cfg[\'workers\'],mp_context=multiprocessing.get_context(\'spawn\')) as pool:\n        engine=League(cfg,pool)\n        try:engine.run()\n        except (TimeoutError,KeyboardInterrupt):\n            engine.save(\'paused_resume_available\')\n            print(\'Paused. Resume the same checkpoint and settings.\',flush=True)\n        except Exception:\n            shutil.copyfile(engine.root/engine.state[\'champion\'],engine.root/\'main.py\')\n            engine.save(\'error_resume_available\');raise\n        print(\'FINAL\',json.dumps(json.loads((engine.root/\'decision.json\').read_text()),indent=2),flush=True)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--config\',required=True)\n    main(parser.parse_args().config)\n', 'support.py': '"""Enhanced Game v2 shared runner/scoring (V13 lineage). No submission API calls."""\nimport argparse, concurrent.futures, contextlib, hashlib, importlib.util, io, json\nimport math, multiprocessing, os, random, shutil, sqlite3, statistics, time, zipfile\nfrom pathlib import Path\nimport optuna\n\nBASE = dict(animals=16,hands=12,land=3,crop_bias=1.5,care_bias=1.3,fert_bias=1.,\n            opponent_weight=0.,liquidate=False,drop_units=5,drop_value=1000000,\n            cash_release=False,deposit_bias=.3,feed_fix=False,care_cap=1.3,\n            plant_floor=12,hire_pace=2,workload_hiring=False,work_per_hand=8,\n            keep_late_hands=True,land_util=.45,land_buffer=700,\n            commitment=1.3,region_weight=.8,distance_weight=.65,dig_value=45,\n            animal_deadline=16,land_deadline=18,expansion_hands=9,night_deposit=True)\nBASE.update(liquidate=True,feed_fix=True,cash_release=True,plant_floor=40)\n# Enhanced Game v2 phase keys. Defaults equal their base key, so an unsearched\n# config reproduces v1 behavior; the league search moves the *_late variants.\nBASE.update(late_day=24,crop_bias_late=BASE[\'crop_bias\'],plant_floor_late=BASE[\'plant_floor\'],deposit_bias_late=BASE[\'deposit_bias\'])\nENV_VERSION = "1.32.7"\nCHECKPOINT_COPY = \'\'\n\ndef write_json(path, obj):\n    path=Path(path);temp=path.with_suffix(path.suffix+".tmp")\n    temp.write_text(json.dumps(obj,indent=2));temp.replace(path)\ndef digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()\ndef load_agent(path, name):\n    spec=importlib.util.spec_from_file_location(name,str(path))\n    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)\n    return module.agent\ndef render(template,params):\n    return template.replace("CFG = {}  # replaced by the trainer","CFG = "+repr(dict(BASE,**params)),1)\ndef worker(job):\n    candidate,opponent,seed,seat,*extras=job\n    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\n        from kaggle_environments import make\n    agent=load_agent(candidate,"candidate");rival=load_agent(opponent,"rival")\n    elapsed=[];route_switches=0;observed_routes=0\n    def timed(obs):\n        nonlocal route_switches,observed_routes\n        old=agent.__globals__.get(\'_MEMORY\',{})\n        previous=old.get(\'jobs\',{}) if old.get(\'tick\')==obs.day*24+obs.hour-1 and obs.hour else {}\n        start=time.perf_counter();a=agent(obs);elapsed.append(time.perf_counter()-start)\n        jobs=agent.__globals__.get(\'_MEMORY\',{}).get(\'jobs\',{})\n        positions=[obs.farms[seat].farmer,*obs.farms[seat].hands]\n        for i,job in previous.items():\n            if i<len(positions) and tuple(positions[i])!=job[0]:\n                observed_routes+=1;route_switches+=jobs.get(i)!=job\n        return a\n    env=make("kaggriculture",configuration={"seed":seed,"episodeSteps":720})\n    env.run([timed,rival] if seat==0 else [rival,timed])\n    final=env.steps[-1]\n    if len(env.steps)!=720 or any(x.status!="DONE" or x.reward is None for x in final):\n        raise RuntimeError("Agent error or incomplete game")\n    if extras and extras[0] and final[seat].reward<final[1-seat].reward:\n        import gzip\n        with gzip.open(extras[0],\'wt\') as f:json.dump(env.toJSON(),f)\n    first80=next((i for i,s in enumerate(env.steps) if s[seat].observation.farms[seat].money>=80000),None)\n    land4=next((i for i,s in enumerate(env.steps) if len(s[seat].observation.farms[seat].unlocked_quadrants)==4),None)\n    snapshots={}\n    action_counts={}\n    for s in env.steps[1:]:\n        a=s[seat].action or {}\n        for u in [a.get(\'farmer\',[]),*a.get(\'hands\',[])]:\n            if u:action_counts[u[0]]=action_counts.get(u[0],0)+1\n    for day in [5,10,15,20,25,28,29]:\n        ob=env.steps[min(day*24+23,719)][seat].observation\n        f=ob.farms[seat];tiles=[t for row in f.tiles for t in row]\n        crops=sum(isinstance(t,dict) and bool(t.get(\'crop\')) for t in tiles)\n        animals=sum(isinstance(t,dict) and bool(t.get(\'animal\')) for t in tiles)\n        active_regions=set((x//5,y//5) for y,row in enumerate(f.tiles) for x,t in enumerate(row)\n                           if isinstance(t,dict) and (t.get(\'crop\') or t.get(\'animal\')))\n        snapshots[str(day)]=dict(cash=f.money,crops=crops,animals=animals,hands=len(f.hands),\n                                quadrants=len(f.unlocked_quadrants),productive_quadrants=len(active_regions),\n                                unwatered=sum(isinstance(t,dict) and bool(t.get(\'crop\')) and not t.get(\'watered_today\') for t in tiles),\n                                unfed=sum(isinstance(t,dict) and bool(t.get(\'animal\')) and not t.get(\'fed_today\') for t in tiles),\n                                carried_units=sum(sum(v.values()) for v in ob.private.inventories),\n                                market_prices=dict(ob.market.prices),town=dict(ob.town),\n                                by_quadrant={str((qx,qy)):sum(isinstance(t,dict) and bool(t.get(\'crop\') or t.get(\'animal\'))\n                                    for y,row in enumerate(f.tiles) for x,t in enumerate(row) if (x//5,y//5)==(qx,qy))\n                                    for qx,qy in [(0,0),(1,0),(0,1),(1,1)]})\n    return dict(seed=seed,seat=seat,opponent=Path(opponent).name,cash=final[seat].reward,\n                rival_cash=final[1-seat].reward,first80=first80,land4=land4,\n                cash_before_last_two_days=env.steps[672][seat].observation.farms[seat].money,\n                max_decision_seconds=max(elapsed),normal=True,snapshots=snapshots,actions=action_counts,\n                route_switches=route_switches,route_continuation_opportunities=observed_routes)\n\ndef score(rows):\n    # Winning dominates; time to $80k is a small secondary preference.\n    terms=[]\n    for r in rows:\n        win=1. if r["cash"]>r["rival_cash"] else (.5 if r["cash"]==r["rival_cash"] else 0.)\n        margin=max(-1.,min(1.,(r["cash"]-r["rival_cash"])/max(3000.,r["rival_cash"])))\n        early=0. if r["first80"] is None else (719-r["first80"])/719\n        # No income ceiling, no reward merely for buying assets. Early cash gets\n        # little weight: excessive early saving can starve future production.\n        terms.append(.65*win+.35*margin+.20*math.log1p(max(0,r[\'cash\'])/80000)+.02*early)\n    return statistics.mean(terms)\n\ndef summary(rows):\n    hits=[r["first80"] for r in rows if r["first80"] is not None]\n    return dict(games=len(rows),mean_cash=statistics.mean(r["cash"] for r in rows),\n                minimum_cash=min(r["cash"] for r in rows),maximum_cash=max(r["cash"] for r in rows),\n                mean_margin=statistics.mean(r["cash"]-r["rival_cash"] for r in rows),\n                win_rate=statistics.mean(r["cash"]>r["rival_cash"] for r in rows),\n                ties=sum(r["cash"]==r["rival_cash"] for r in rows),\n                reach_80k_rate=len(hits)/len(rows),\n                median_first_80k_turn_among_hits=statistics.median(hits) if hits else None,\n                earliest_80k_turn=min(hits) if hits else None,\n                four_quadrants_rate=statistics.mean(r["land4"] is not None for r in rows),\n                mean_last_two_days_cash_gain=statistics.mean(r["cash"]-r["cash_before_last_two_days"] for r in rows),\n                max_decision_seconds=max(r["max_decision_seconds"] for r in rows))\n\ndef seed_margin_interval(rows):\n    by={}\n    for r in rows:by.setdefault(r["seed"],[]).append(r["cash"]-r["rival_cash"])\n    means=[statistics.mean(v) for v in by.values()]\n    rng=random.Random(130)\n    boot=sorted(statistics.mean(rng.choices(means,k=len(means))) for _ in range(2000))\n    return [boot[50],boot[1949]]\n\ndef sample(trial):\n    p=dict(BASE)\n    p.update(\n        animals=trial.suggest_int("animals",12,20,step=2),\n        hands=trial.suggest_int("hands",10,12),\n        land=trial.suggest_int("land",3,4),\n        crop_bias=trial.suggest_float("crop_bias",.6,2.5),\n        care_bias=trial.suggest_float("care_bias",.6,2.),\n        fert_bias=trial.suggest_float("fert_bias",.6,1.8),\n        opponent_weight=trial.suggest_categorical("opponent_weight",[0.,.5,1.]),\n        liquidate=True,\n        drop_units=trial.suggest_int("drop_units",2,6),\n        drop_value=trial.suggest_categorical("drop_value",[250,500,1000,1000000]),\n        cash_release=trial.suggest_categorical("cash_release",[False,True]),\n        deposit_bias=trial.suggest_float("deposit_bias",.15,.65),\n        feed_fix=True,\n        care_cap=trial.suggest_categorical("care_cap",[1.3,2.,3.]),\n        plant_floor=trial.suggest_categorical("plant_floor",[12,25,40,60,80]),\n        hire_pace=trial.suggest_int("hire_pace",1,3),\n        workload_hiring=trial.suggest_categorical("workload_hiring",[False,True]),\n        work_per_hand=trial.suggest_categorical("work_per_hand",[6,8,10,12]),\n        keep_late_hands=True,\n        land_util=trial.suggest_categorical("land_util",[.35,.45,.55,.75]),\n        land_buffer=trial.suggest_categorical("land_buffer",[300,700,1500]),\n        commitment=trial.suggest_categorical(\'commitment\',[1.,1.3,1.7]),\n        region_weight=trial.suggest_categorical(\'region_weight\',[.65,.8,1.]),\n        distance_weight=trial.suggest_categorical(\'distance_weight\',[.45,.65,.85]),\n        dig_value=trial.suggest_categorical(\'dig_value\',[20,45,75]),\n        animal_deadline=trial.suggest_categorical(\'animal_deadline\',[12,16,19]),\n        expansion_hands=trial.suggest_categorical(\'expansion_hands\',[7,9,11]),\n        night_deposit=trial.suggest_categorical(\'night_deposit\',[False,True]))\n    return p\n\nclass Runner:\n    def __init__(self,root,cfg,pool):\n        self.root=root;self.cfg=cfg;self.pool=pool;self.cache={}\n        self.started=time.monotonic();self.deadline=self.started+cfg["time_budget_minutes"]*60\n        self.cache_path=root/"games.jsonl"\n        if self.cache_path.exists():\n            for line in self.cache_path.read_text().splitlines():\n                try:\n                    x=json.loads(line);self.cache[x["key"]]=x["result"]\n                except json.JSONDecodeError: pass # interrupted final append\n    def key(self,candidate,opponent,seed,seat):\n        return ":".join([digest(candidate),digest(opponent),str(seed),str(seat),ENV_VERSION])\n    def evaluate(self,candidate,opponents,seeds):\n        rows=[];batch=[];keys=[]\n        for seed in seeds:\n            for opponent in opponents:\n                for seat in [0,1]:\n                    k=self.key(candidate,opponent,seed,seat)\n                    if k in self.cache:\n                        row=dict(self.cache[k]);row["opponent"]=opponent.name;rows.append(row)\n                    else:\n                        replay=None\n                        if 19000000<=seed<19600000 and (seed-19000000)%10000<2:\n                            replay=str(self.root/\'replays\'/f\'{candidate.stem}_{opponent.stem}_{seed}_{seat}.json.gz\')\n                        keys.append(k);batch.append((str(candidate),str(opponent),seed,seat,replay))\n        chunk=max(2,self.cfg["workers"]*2)\n        for i in range(0,len(batch),chunk):\n            if time.monotonic()>=self.deadline:raise TimeoutError("Run time budget reached")\n            results=list(self.pool.map(worker,batch[i:i+chunk]))\n            with self.cache_path.open("a") as f:\n                for k,r in zip(keys[i:i+chunk],results):\n                    self.cache[k]=r;rows.append(r)\n                    f.write(json.dumps({"key":k,"result":r})+"\\n")\n                f.flush();os.fsync(f.fileno())\n            if len(batch)>16:print("  evaluated",min(i+chunk,len(batch)),"/",len(batch),"games",flush=True)\n        return rows\n\ndef checkpoint(root):\n    # SQLite backup avoids copying an inconsistent live database.\n    db=root/"study.db"\n    if db.exists():\n        with sqlite3.connect(str(db)) as src, sqlite3.connect(str(root/"study-backup.db")) as dest:src.backup(dest)\n    path=root/"checkpoint.zip";temp=root/"checkpoint.zip.tmp"\n    with zipfile.ZipFile(temp,"w",zipfile.ZIP_DEFLATED) as archive:\n        names=["state.json","manifest.json","champion.py","main.py","games.jsonl","decision.json"]\n        for name in names:\n            p=root/name\n            if p.exists():archive.write(p,name)\n        if (root/"study-backup.db").exists():archive.write(root/"study-backup.db","study.db")\n        for folder in ["trials","reports","replays"]:\n            for p in (root/folder).glob("*"):\n                if p.is_file():archive.write(p,str(p.relative_to(root)))\n    temp.replace(path)\n    if CHECKPOINT_COPY:\n        mirror=Path(CHECKPOINT_COPY).expanduser()\n        mirror.parent.mkdir(parents=True,exist_ok=True)\n        if mirror.resolve()!=path.resolve():\n            try:\n                pending=mirror.with_suffix(mirror.suffix+\'.tmp\')\n                shutil.copyfile(path,pending);pending.replace(mirror)\n            except OSError as e:\n                print(\'Checkpoint mirror failed; local checkpoint is intact:\',str(e),flush=True)\n\ndef verify_export(root,opponent,seed):\n    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):\n        from kaggle_environments import make\n    e=make("kaggriculture",configuration={"seed":seed,"episodeSteps":720})\n    e.run([str(root/"main.py"),str(opponent)])\n    if len(e.steps)!=720 or any(x.status!="DONE" or x.reward is None for x in e.steps[-1]):\n        raise RuntimeError("Exported file failed its loading test")\n    # Preserve an inspectable replay, not only an aggregate score.\n    import gzip\n    (root/\'replays\').mkdir(exist_ok=True)\n    with gzip.open(root/\'replays\'/f\'filecheck_{seed}.json.gz\',\'wt\') as f:\n        json.dump(e.toJSON(),f)\n    return {"cash":[s.reward for s in e.steps[-1]],"status":[s.status for s in e.steps[-1]]}\n\ndef main(config_path):\n    global CHECKPOINT_COPY\n    cfg=json.loads(Path(config_path).read_text());root=Path(cfg["output_dir"]).resolve()\n    CHECKPOINT_COPY=cfg.get(\'checkpoint_copy\',\'\')\n    root.mkdir(parents=True,exist_ok=True)\n    source=Path(__file__).resolve().parent\n    for name in ["trials","reports","opponents","replays"]:(root/name).mkdir(exist_ok=True)\n    template=(source/"policy_template.py").read_text()\n    for version in ["v9","v11","v12"]:shutil.copyfile(source/f"main_{version}.py",root/"opponents"/f"main_{version}.py")\n    for name,params in {\n        "crop_heavy":dict(BASE,animals=6,hands=12,land=4,crop_bias=2.,plant_floor=60,land_util=.35),\n        "livestock_heavy":dict(BASE,animals=20,hands=12,care_bias=1.8,feed_fix=True),\n        "regional_mixed":dict(BASE,land=4,region_weight=.65,commitment=1.7),\n        "reactive_market":dict(BASE,opponent_weight=.5,care_cap=3.,region_weight=1.,commitment=1.)}.items():\n        (root/"opponents"/(name+".py")).write_text(render(template,params))\n    # Extra executable opponents can be attached as a private Kaggle input.\n    extras=[]\n    for p in cfg.get("extra_opponents",[]):\n        path=Path(p);target=root/"opponents"/("extra_"+digest(path)[:12]+".py")\n        shutil.copyfile(path,target);extras.append(target)\n    opponents=sorted((root/\'opponents\').glob(\'*.py\'))\n    with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):\n        import kaggle_environments\n    if kaggle_environments.__version__!=ENV_VERSION:raise RuntimeError("Wrong simulator version")\n    signature=dict(template_sha=digest(source/"policy_template.py"),\n                   trainer_sha=digest(Path(__file__)),opponents={p.name:digest(p) for p in opponents},\n                   environment=ENV_VERSION,train_seed_count=cfg["train_seed_count"],\n                   holdout_seed_count=cfg[\'holdout_seed_count\'],v9_gate=cfg.get(\'v9_win_rate\',.80))\n    manifest=root/"manifest.json"\n    if manifest.exists() and json.loads(manifest.read_text())!=signature:\n        raise ValueError("Checkpoint belongs to a different search definition; use a new output directory")\n    write_json(manifest,signature)\n    state_path=root/"state.json"\n    if state_path.exists():state=json.loads(state_path.read_text())\n    else:\n        state={"next_cycle":0,"tested_hashes":[],"champion":"v9","champion_sha":digest(root/"opponents/main_v9.py")}\n        shutil.copyfile(root/"opponents/main_v9.py",root/"champion.py")\n        write_json(state_path,state)\n    if digest(root/"champion.py")!=state["champion_sha"]:raise ValueError("Checkpoint champion hash mismatch")\n    shutil.copyfile(root/"champion.py",root/"main.py")\n    write_json(root/"decision.json",{"export":state["champion"],"promoted":False,"status":"search_in_progress",\n                                    "sha256":digest(root/"main.py")})\n    optuna.logging.set_verbosity(optuna.logging.WARNING)\n    study=optuna.create_study(study_name="kaggriculture-v13",direction="maximize",\n        storage="sqlite:///"+str(root/"study.db"),load_if_exists=True,\n        sampler=optuna.samplers.TPESampler(seed=20260916+state["next_cycle"]),\n        pruner=optuna.pruners.MedianPruner(n_startup_trials=12,n_warmup_steps=1))\n    for t in study.get_trials(deepcopy=False):\n        if t.state==optuna.trial.TrialState.RUNNING:\n            study.tell(t.number,state=optuna.trial.TrialState.FAIL)\n    if not study.trials:\n        for p in [BASE,dict(BASE,commitment=1.,region_weight=1.),\n                  dict(BASE,land=4,land_util=.35),\n                  dict(BASE,drop_value=1000,night_deposit=False,workload_hiring=True)]:\n            study.enqueue_trial(p)\n    checkpoint(root)\n    context=multiprocessing.get_context("spawn")\n    with concurrent.futures.ProcessPoolExecutor(max_workers=cfg["workers"],mp_context=context) as pool:\n        runner=Runner(root,cfg,pool)\n        search_deadline=runner.started+cfg["time_budget_minutes"]*60*cfg["search_fraction"]\n        def objective(trial):\n            params=sample(trial);candidate=root/"trials"/f"v13_trial_{trial.number:05d}.py"\n            candidate.write_text(render(template,params))\n            rows=[]\n            for index in range(cfg["train_seed_count"]):\n                if time.monotonic()>=search_deadline:raise optuna.TrialPruned("Search budget reached")\n                rows+=runner.evaluate(candidate,opponents,[17000000+index])\n                trial.report(score(rows),index)\n                if trial.should_prune():raise optuna.TrialPruned()\n            trial.set_user_attr("summary",summary(rows))\n            trial.set_user_attr("sha256",digest(candidate))\n            write_json(candidate.with_suffix(".json"),{"parameters":params,"summary":summary(rows),"games":rows})\n            return score(rows)\n        def completed(study,trial):\n            checkpoint(root)\n            value=round(trial.value,4) if trial.value is not None else None\n            print("Trial",trial.number,trial.state.name,"score",value,\n                  "elapsed_minutes",round((time.monotonic()-runner.started)/60,1),flush=True)\n        remaining=max(0,cfg["total_trials"]-len(study.trials))\n        # Queued warm starts already count as WAITING, but must still be executed.\n        remaining+=sum(t.state==optuna.trial.TrialState.WAITING for t in study.trials)\n        try:\n            if not state.get(\'pending_holdout\') and not cfg.get(\'evaluate_only\',False):\n                study.optimize(objective,n_trials=remaining,timeout=max(0,search_deadline-time.monotonic()),\n                           callbacks=[completed],catch=(RuntimeError,))\n        except TimeoutError:\n            print("Search stopped at time budget; retaining safe export.",flush=True)\n        complete=sorted((t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE),\n                        key=lambda t:t.value,reverse=True)\n        candidates=[];seen=set(state["tested_hashes"])\n        for t in complete:\n            path=root/"trials"/f"v13_trial_{t.number:05d}.py";sha=digest(path)\n            if sha not in seen:\n                candidates.append(path);seen.add(sha)\n            if len(candidates)>=cfg["finalists"]:break\n        if state.get(\'pending_holdout\'):\n            candidates=[root/\'trials\'/state[\'pending_holdout\'][\'candidate\']]\n        decision={"export":state["champion"],"promoted":False,"status":"no_new_completed_candidate",\n                  "sha256":digest(root/"main.py")}\n        try:\n            if candidates:\n                validation=[]\n                for candidate in ([] if state.get(\'pending_holdout\') else candidates):\n                    print("Validation:",candidate.name,flush=True)\n                    rows=runner.evaluate(candidate,opponents,range(18000000,18000000+cfg["validation_seed_count"]))\n                    record={"candidate":candidate.name,"score":score(rows),"summary":summary(rows),"games":rows}\n                    write_json(root/"reports"/("validation_"+candidate.stem+".json"),record)\n                    validation.append((record["score"],candidate))\n                    checkpoint(root)\n                if state.get(\'pending_holdout\'):\n                    winner=candidates[0];cycle=state[\'pending_holdout\'][\'cycle\']\n                else:\n                    _,winner=max(validation,key=lambda x:x[0])\n                    cycle=state["next_cycle"];state["next_cycle"]+=1\n                    state[\'pending_holdout\']={\'candidate\':winner.name,\'cycle\':cycle}\n                    state["tested_hashes"].append(digest(winner));write_json(state_path,state);checkpoint(root)\n                holdout_opponents=list(opponents)\n                if digest(root/"champion.py") not in [digest(p) for p in holdout_opponents]:\n                    holdout_opponents=holdout_opponents+[root/"champion.py"]\n                seeds=range(19000000+cycle*10000,19000000+cycle*10000+cfg["holdout_seed_count"])\n                print("Fresh holdout:",winner.name,"cycle",cycle,flush=True)\n                rows=runner.evaluate(winner,holdout_opponents,seeds)\n                per={p.name:summary([r for r in rows if r["opponent"]==p.name]) for p in holdout_opponents}\n                interval=seed_margin_interval(rows)\n                per_intervals={p.name:seed_margin_interval([r for r in rows if r[\'opponent\']==p.name]) for p in holdout_opponents}\n                passed=(len(seeds)>=24 and interval[0]>0 and\n                        per[\'main_v9.py\'][\'win_rate\']>=cfg.get(\'v9_win_rate\',.80) and\n                        per_intervals[\'main_v9.py\'][0]>0 and\n                        all(x["mean_margin"]>0 and x["win_rate"]>=.55 for x in per.values()) and\n                        all(r[\'max_decision_seconds\']<.8 for r in rows))\n                report={"candidate":winner.name,"sha256":digest(winner),"summary":summary(rows),\n                        "per_opponent":per,"per_opponent_margin_intervals":per_intervals,"seed_bootstrap_margin_interval":interval,"passed":passed,"games":rows,\n                        "note":"Local baselines, not actual leaderboard leaders. Both seats share a seed cluster."}\n                write_json(root/"reports"/f"holdout_cycle_{cycle:03d}.json",report)\n                decision={"export":state["champion"],"promoted":False,"status":"holdout_failed",\n                          "challenger":winner.name,"holdout":report["summary"],"sha256":digest(root/"main.py")}\n                if passed:\n                    shutil.copyfile(winner,root/"main.py")\n                    try:report["file_loading_check"]=verify_export(root,root/"opponents/main_v9.py",9500000+cycle)\n                    except Exception:\n                        shutil.copyfile(root/"champion.py",root/"main.py");raise\n                    shutil.copyfile(winner,root/"champion.py")\n                    state.update(champion=winner.stem,champion_sha=digest(winner))\n                    write_json(state_path,state)\n                    decision.update(export=state["champion"],promoted=True,status="holdout_passed",\n                                    sha256=digest(root/"main.py"))\n                    write_json(root/"reports"/f"holdout_cycle_{cycle:03d}.json",report)\n                state.pop(\'pending_holdout\',None);write_json(state_path,state)\n                write_json(root/\'decision.json\',decision);checkpoint(root)\n                # Self-play is diagnostic, not a promotion criterion.\n                if time.monotonic()<runner.deadline-60:\n                    rows=runner.evaluate(root/"main.py",[root/"main.py"],range(19600000+cycle*100,19600000+cycle*100+cfg["selfplay_seed_count"]))\n                    write_json(root/"reports"/f"selfplay_cycle_{cycle:03d}.json",{"summary":summary(rows),"games":rows})\n        except TimeoutError:\n            if not decision.get(\'promoted\'):\n                decision["status"]="time_budget_reached_keep_incumbent"\n        except Exception as error:\n            decision["status"]="evaluation_error_keep_incumbent"\n            decision["error"]=type(error).__name__+": "+str(error)\n            shutil.copyfile(root/"champion.py",root/"main.py")\n        write_json(root/"decision.json",decision);checkpoint(root)\n        print("FINAL:",json.dumps(decision,indent=2),flush=True)\n    print("Upload file:",root/"main.py",flush=True)\n    print("Resume archive:",root/"checkpoint.zip",flush=True)\n\nif __name__=="__main__":\n    parser=argparse.ArgumentParser();parser.add_argument("--config",required=True)\n    args=parser.parse_args();main(args.config)\n', 'policy_template.py': '"""Enhanced Game v2: phase-dependent league-trained farming policy. Standard library only."""\nimport copy\n\n# A soft commitment, never an unconditional cached action. Reset every day/game.\n_MEMORY = {}\n\nCFG = {}  # replaced by the trainer\nCROPS={\'WHEAT\':(10,2,4,4),\'CARROT\':(20,2,3,3),\'MELON\':(80,10,10,6),\'TOMATO\':(50,8,11,4),\'STRAWBERRY\':(100,10,16,4)}\nANIMALS={\'COW\':(400,\'MILK\',8,2),\'SHEEP\':(500,\'WOOL\',6,3),\'GOOSE\':(300,\'EGG\',4,1)}\nBASE={\'WHEAT\':25,\'CARROT\':35,\'MELON\':250,\'TOMATO\':60,\'STRAWBERRY\':120,\'MILK\':160,\'WOOL\':200,\'EGG\':50,\'FERTILIZER\':100}\n\ndef agent(obs):\n    global _MEMORY\n    tick=obs[\'day\']*24+obs[\'hour\']; player=obs[\'player\']\n    if _MEMORY.get(\'tick\')!=tick-1 or _MEMORY.get(\'player\')!=player or obs[\'hour\']==0:\n        _MEMORY={}\n    old_jobs=_MEMORY.get(\'jobs\',{}); jobs={}\n    me=copy.deepcopy(obs[\'farms\'][obs[\'player\']]);pr=copy.deepcopy(obs[\'private\'])\n    tiles=me[\'tiles\'];n=len(tiles);day=obs[\'day\'];hour=obs[\'hour\'];prices=obs[\'market\'][\'prices\'];remaining=30-day\n    # Phase-dependent effective config. Overlays _mid/_late variants of a few\n    # high-leverage keys once the game crosses day thresholds. Absent variants\n    # fall back to the base key, so a config without them reproduces v1 exactly.\n    P=dict(CFG)\n    _phased=(\'crop_bias\',\'plant_floor\',\'deposit_bias\',\'care_bias\')\n    if day>=CFG.get(\'mid_day\',12):\n        for _k in _phased:\n            if _k+\'_mid\' in CFG: P[_k]=CFG[_k+\'_mid\']\n    if day>=CFG.get(\'late_day\',24):\n        for _k in _phased:\n            if _k+\'_late\' in CFG: P[_k]=CFG[_k+\'_late\']\n    shed=pr[\'shed\'];seeds=pr[\'seeds\'];invs=pr[\'inventories\'];positions=[me[\'farmer\']]+me[\'hands\']\n    center=n//2;access=[(center-1,center-1),(center,center-1),(center-1,center),(center,center)]\n    def dist(a,b):return abs(a[0]-b[0])+abs(a[1]-b[1])\n    def move(a,b):\n        dx,dy=b[0]-a[0],b[1]-a[1]\n        return [\'EAST\' if dx>0 else \'WEST\'] if abs(dx)>=abs(dy) and dx else [\'SOUTH\' if dy>0 else \'NORTH\']\n    coords=[(x,y) for y in range(n) for x in range(n) if tiles[y][x]!=\'LOCKED\']\n    animals=[(x,y) for x,y in coords if isinstance(tiles[y][x],dict) and tiles[y][x].get(\'animal\')]\n    counts={c:sum(isinstance(tiles[y][x],dict) and tiles[y][x].get(\'crop\')==c for x,y in coords) for c in CROPS}\n    nlive=len(animals);claimed=set();actions=[]\n    other=obs[\'farms\'][1-obs[\'player\']]\n    opponent_crops={c:sum(isinstance(t,dict) and t.get(\'crop\')==c for row in other[\'tiles\'] for t in row) for c in CROPS}\n    occupied=sum(isinstance(tiles[y][x],dict) and bool(tiles[y][x].get(\'crop\') or tiles[y][x].get(\'animal\')) for x,y in coords)\n    regions=sorted(set((x//center,y//center) for x,y in coords))\n    # Stable spatial preference, with cross-region emergency work still allowed.\n    homes={i:regions[i%len(regions)] for i in range(len(positions))}\n\n    # Planting estimates account for crops already growing, maturity and expected market saturation.\n    def crop_value(c):\n        cost,first,peak,yld=CROPS[c]\n        if day+first>=29 or (hour>=21 and c!=\'WHEAT\'):return -1\n        lifespan=min(peak,remaining-2)\n        if c in (\'TOMATO\',\'STRAWBERRY\'):\n            interval=1 if c==\'TOMATO\' else 2;yld=max(0,min(4,1+(remaining-2-first)//interval))\n        price=prices[c]\n        if CFG[\'opponent_weight\']>0:price=max(BASE[c]*.15,price-CFG[\'opponent_weight\']*opponent_crops[c]*BASE[c]*.06)\n        if c in (\'MELON\',\'STRAWBERRY\'):price=max(BASE[c]*.20,price-counts[c]*(18 if c==\'MELON\' else 12))\n        if c==\'WHEAT\' and nlive:price=max(price,32)\n        style=CFG.get(\'crop_style\',\'balanced\')\n        style_weight=1.0\n        if style==\'quick\':style_weight=1.5 if c in (\'WHEAT\',\'CARROT\') else .75\n        if style==\'orchard\':style_weight=1.5 if c in (\'TOMATO\',\'STRAWBERRY\') else .75\n        return (yld*price-cost)/(lifespan+4)*P[\'crop_bias\']*style_weight\n    crop_scores={c:crop_value(c) for c in CROPS}\n    for idx,pos0 in enumerate(positions):\n        pos=tuple(pos0);inv=invs[idx];best=(-1,None,None);carried=next((a for a in ANIMALS if inv.get(a,0)),None)\n        nearest_shed=min(access,key=lambda q:dist(pos,q));ds=dist(pos,nearest_shed)\n        def offer(value,target,act,key=None):\n            nonlocal best\n            if key is not None and key in claimed:return\n            distance=dist(pos,target)\n            if day==29 and CFG[\'liquidate\']:\n                if act[0] not in (\'HARVEST\',\'COLLECT_FERTILIZER\',\'DROP\'):return\n                delivery=0 if act[0]==\'DROP\' else min(dist(target,s) for s in access)+1\n                if distance+1+delivery>23-hour:return\n            if distance>23-hour:return\n            score=value/(1+distance*CFG.get(\'distance_weight\',.85))\n            if key and day<29:\n                if (target[0]//center,target[1]//center)!=homes[idx]:\n                    score*=CFG.get(\'region_weight\',1.)\n                previous=old_jobs.get(idx)\n                if previous and previous==(target,tuple(act)):\n                    score*=CFG.get(\'commitment\',1.)\n            if score>best[0]:best=(score,target,(act,key))\n        # Deposits make produce sellable before night and before the final turn.\n        goods=sum(v for k,v in inv.items() if k not in (\'WHEAT\',\'FERTILIZER\') and k not in ANIMALS)\n        goods_value=sum(v*prices.get(k,0) for k,v in inv.items() if k not in ANIMALS and k!=\'WHEAT\')\n        deposit=(day==29 and hour+ds>=19 and sum(inv.values())>0) or goods>=CFG[\'drop_units\'] or inv.get(\'FERTILIZER\',0)>=4\n        deposit=deposit or goods_value>=CFG[\'drop_value\'] or (CFG[\'cash_release\'] and me[\'money\']<100 and goods_value>0)\n        # Inventory is automatically deposited each night. Avoid unnecessary trips\n        # when solvent and overnight storage is safe; final day has no such sale.\n        if CFG.get(\'night_deposit\',False) and day<29 and me[\'money\']>=500:\n            total_stock=sum(shed.values())+sum(sum(v.values()) for v in invs)\n            if total_stock<80:deposit=False\n        if day==29 and CFG[\'liquidate\'] and sum(inv.values())>0:deposit=True\n        if deposit and not carried:\n            urgency=3000 if day==29 and hour+ds>=20 else max(90,goods_value*P[\'deposit_bias\'])\n            offer(urgency,nearest_shed,[\'DROP\'])\n        if carried:\n            kind=\'COOP\' if carried==\'GOOSE\' else \'PASTURE\'\n            for x,y in coords:\n                t=tiles[y][x]\n                if t is None:offer(280-5*min(dist((x,y),s) for s in access),(x,y),[\'BUILD_COOP\' if kind==\'COOP\' else \'BUILD_PASTURE\'],(\'tile\',x,y))\n                elif isinstance(t,dict) and t.get(\'kind\')==kind and not t.get(\'animal\'):offer(600,(x,y),[\'PLACE\',carried,1],(\'tile\',x,y))\n        else:\n            pending=next((a for a in ANIMALS if shed.get(a,0)>0),None)\n            if pending and day<20:offer(350,nearest_shed,[\'PICKUP\',pending,1])\n        needed=sum(not tiles[y][x].get(\'fed_today\') for x,y in animals)\n        held_feed=sum(v.get(\'WHEAT\',0) for v in invs)\n        if nlive and inv.get(\'WHEAT\',0)==0 and shed.get(\'WHEAT\',0)>0 and hour<22 and (not CFG[\'feed_fix\'] or (day<29 and needed>held_feed)):\n            offer(210,nearest_shed,[\'PICKUP\',\'WHEAT\',min(4,shed[\'WHEAT\'])])\n        for x,y in coords:\n            t=tiles[y][x];target=(x,y);key=(\'tile\',x,y)\n            if key in claimed:continue\n            if isinstance(t,dict) and t.get(\'animal\'):\n                species=t[\'animal\'];cost,product,first,interval=ANIMALS[species];price=prices[product]\n                if not t.get(\'fed_today\') and inv.get(\'WHEAT\',0):offer(190+130*t.get(\'consecutive_unfed\',0)+hour*3,target,[\'FEED\'],key)\n                if t.get(\'yield_units\',0):offer(60+t[\'yield_units\']*max(price,10)*.8,target,[\'HARVEST\'],key)\n                if not t.get(\'cared_today\') and day<29 and t.get(\'yield_units\',0)<5:offer(max(12,min(price,BASE[product]*CFG[\'care_cap\']))*P[\'care_bias\'],target,[\'CARE\'],key)\n                if t.get(\'fertilizer_available\'):offer(max(40,prices[\'FERTILIZER\'])*.85,target,[\'COLLECT_FERTILIZER\'],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'PLANT\':\n                c=t[\'crop\'];cost,first,peak,yld=CROPS[c];age=day-t[\'planted_day\'];yield_units=t.get(\'yield_units\',0)\n                fertile=t.get(\'fertilized_until_day\',-1)>=day\n                needs_fert=(c in (\'TOMATO\',\'STRAWBERRY\') and age>=first-2) or (c not in (\'TOMATO\',\'STRAWBERRY\') and age>=max(0,first-1) and age<peak)\n                if not fertile and needs_fert and inv.get(\'FERTILIZER\',0) and day<29:\n                    offer(min(200,prices[c]*1.4)*CFG[\'fert_bias\'],target,[\'FERTILIZE\'],key)\n                ready=yield_units>0 and age>=first and (c in (\'TOMATO\',\'STRAWBERRY\') or age>=peak or yield_units>=6 or day==29)\n                if ready:offer(100+yield_units*prices[c]*.55+(200 if day==29 else 0),target,[\'HARVEST\'],key)\n                if not t.get(\'watered_today\') and not (day==29 and ready):offer(95+hour*4+120*t.get(\'consecutive_unwatered\',0),target,[\'WATER\'],key)\n            elif t is None and not carried and hour<22:\n                available=[c for c in CROPS if seeds.get(c,0)>0 and crop_scores[c]>0]\n                if available:\n                    c=max(available,key=lambda c:crop_scores[c]);offer(max(P[\'plant_floor\'],crop_scores[c]),target,[\'PLANT\',c],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'WEED\' and day<26:offer(CFG.get(\'dig_value\',20),target,[\'DIG\'],key)\n        if best[1] is None:action=[\'PASS\']\n        else:\n            _,target,(task,key)=best\n            if key:claimed.add(key)\n            if pos!=target:\n                action=move(pos,target);jobs[idx]=(target,tuple(task))\n            else:\n                action=task;op=action[0];x,y=pos;t=tiles[y][x]\n                # Project same-turn changes to coordinate later units and market orders.\n                if op==\'PLANT\':\n                    seeds[action[1]]-=1;tiles[y][x]={\'kind\':\'PLANT\',\'crop\':action[1],\'planted_day\':day,\'yield_units\':0,\'watered_today\':False}\n                elif op==\'DIG\':tiles[y][x]=None\n                elif op==\'WATER\':t[\'watered_today\']=True\n                elif op==\'FEED\':t[\'fed_today\']=True;inv[\'WHEAT\']-=1\n                elif op==\'CARE\':t[\'cared_today\']=True\n                elif op==\'FERTILIZE\':t[\'fertilized_until_day\']=day+2;inv[\'FERTILIZER\']-=1\n                elif op==\'COLLECT_FERTILIZER\':t[\'fertilizer_available\']=False;inv[\'FERTILIZER\']=inv.get(\'FERTILIZER\',0)+1\n                elif op==\'HARVEST\':\n                    product=ANIMALS[t[\'animal\']][1] if t.get(\'animal\') else t[\'crop\'];inv[product]=inv.get(product,0)+t[\'yield_units\'];t[\'yield_units\']=0\n                    if t.get(\'crop\') in (\'WHEAT\',\'CARROT\',\'MELON\'):tiles[y][x]=None\n                elif op.startswith(\'BUILD_\'):tiles[y][x]={\'kind\':op[6:]}\n                elif op==\'PLACE\' and action[1] in ANIMALS:inv[action[1]]-=1;tiles[y][x]={\'kind\':\'COOP\' if action[1]==\'GOOSE\' else \'PASTURE\',\'animal\':action[1],\'yield_units\':0,\'placed_day\':day}\n                elif op==\'PICKUP\':\n                    k=action[1];amount=min(action[2],shed.get(k,0));shed[k]-=amount;inv[k]=inv.get(k,0)+amount\n                elif op==\'DROP\':\n                    room=max(0,100-sum(shed.values()))\n                    for k,v in list(inv.items()):\n                        take=min(room,v);shed[k]=shed.get(k,0)+take;room-=take;inv[k]=0\n        actions.append(action)\n    orders=[];cash=me[\'money\'];reserve_feed=0 if day==29 else nlive*2\n    for k,q in shed.items():\n        if k in ANIMALS:continue\n        sell=max(0,q-(reserve_feed if k==\'WHEAT\' else 0))\n        if sell:orders.append([\'SELL\',k,sell])\n    # Spend only money already in the bank; each planned purchase reserves its cost.\n    def buy(order,cost):\n        nonlocal cash\n        if cost<=cash and len(orders)<10:orders.append(order);cash-=cost;return True\n        return False\n    target_hands=min(CFG[\'hands\'],4+day//CFG[\'hire_pace\'])\n    if CFG[\'workload_hiring\']:\n        workload=sum(counts.values())+nlive*3\n        target_hands=min(CFG[\'hands\'],max(4,2+int(workload/CFG[\'work_per_hand\'])))\n    if day>=28:target_hands=CFG[\'hands\'] if CFG[\'keep_late_hands\'] else max(5,CFG[\'hands\']-1)\n    fib=[1,1,2,3,5,8,13,21,34,55,89,144]\n    for h in range(me[\'hires_today\'],target_hands):\n        if hour>12 or cash<fib[h]+40:break\n        buy([\'HIRE\'],fib[h])\n    # Include animals placed earlier in this same planned turn; otherwise one\n    # disappears from the count and can trigger an unintended extra purchase.\n    projected_live=sum(isinstance(tiles[y][x],dict) and bool(tiles[y][x].get(\'animal\')) for x,y in coords)\n    total_animals=projected_live+sum(shed.get(a,0) for a in ANIMALS)+sum(inv.get(a,0) for inv in invs for a in ANIMALS)\n    if day<CFG.get(\'animal_deadline\',12) and total_animals<min(CFG[\'animals\'],5+day):\n        populations={a:sum(tiles[y][x].get(\'animal\')==a for x,y in animals)+shed.get(a,0)+sum(v.get(a,0) for v in invs) for a in ANIMALS}\n        values={a:(max(prices[ANIMALS[a][1]],BASE[ANIMALS[a][1]]*.2)*(1+ANIMALS[a][3])/ANIMALS[a][3])/(1+populations[a]*.35) for a in ANIMALS}\n        preference={\'milk\':\'COW\',\'wool\':\'SHEEP\',\'eggs\':\'GOOSE\'}.get(CFG.get(\'animal_style\',\'balanced\'))\n        if preference:values[preference]*=1.8\n        species=max(values,key=values.get)\n        if cash>ANIMALS[species][0]+200:buy([\'BUY_ANIMAL\',species,1],ANIMALS[species][0])\n    if nlive and hour<22 and (day<29 or not CFG[\'liquidate\']):\n        deficit=max(0,nlive*2-shed.get(\'WHEAT\',0)-sum(v.get(\'WHEAT\',0) for v in invs));qty=min(deficit,int(max(0,cash-50)//(prices[\'WHEAT\']+2)))\n        if qty:buy([\'BUY_PRODUCT\',\'WHEAT\',qty],qty*(prices[\'WHEAT\']+2))\n    for c in sorted(CROPS,key=lambda c:-crop_scores[c]):\n        if crop_scores[c]>0 and seeds.get(c,0)<2 and cash>100:\n            qty=2-seeds.get(c,0);buy([\'BUY_SEED\',c,qty],CROPS[c][0]*qty)\n    landcount=len(me[\'unlocked_quadrants\'])\n    # Third/fourth quadrants require occupancy and staffing, not cash alone.\n    staffed=(landcount<2 or len(positions)>=CFG.get(\'expansion_hands\',9))\n    if landcount<CFG[\'land\'] and day<CFG.get(\'land_deadline\',18) and staffed and occupied/max(1,len(coords))>=CFG[\'land_util\']:\n        cost=[1000,2000,4000][landcount-1]\n        if cash>cost+CFG[\'land_buffer\']:buy([\'BUY_LAND\'],cost)\n    _MEMORY={\'tick\':tick,\'player\':player,\'jobs\':jobs}\n    return {\'farmer\':actions[0],\'hands\':actions[1:],\'market\':orders[:10]}\n', 'main_v9.py': '"""Kaggriculture: coordinated farming policy. Standard library only."""\nimport copy\n\nCFG={\'animals\':12,\'hands\':10,\'land\':3,\'crop_bias\':1.0,\'care_bias\':1.0,\'fert_bias\':1.0}\nCROPS={\'WHEAT\':(10,2,4,4),\'CARROT\':(20,2,3,3),\'MELON\':(80,10,10,6),\'TOMATO\':(50,8,11,4),\'STRAWBERRY\':(100,10,16,4)}\nANIMALS={\'COW\':(400,\'MILK\',8,2),\'SHEEP\':(500,\'WOOL\',6,3),\'GOOSE\':(300,\'EGG\',4,1)}\nBASE={\'WHEAT\':25,\'CARROT\':35,\'MELON\':250,\'TOMATO\':60,\'STRAWBERRY\':120,\'MILK\':160,\'WOOL\':200,\'EGG\':50,\'FERTILIZER\':100}\n\ndef agent(obs):\n    me=copy.deepcopy(obs[\'farms\'][obs[\'player\']]);pr=copy.deepcopy(obs[\'private\'])\n    tiles=me[\'tiles\'];n=len(tiles);day=obs[\'day\'];hour=obs[\'hour\'];prices=obs[\'market\'][\'prices\'];remaining=30-day\n    shed=pr[\'shed\'];seeds=pr[\'seeds\'];invs=pr[\'inventories\'];positions=[me[\'farmer\']]+me[\'hands\']\n    center=n//2;access=[(center-1,center-1),(center,center-1),(center-1,center),(center,center)]\n    def dist(a,b):return abs(a[0]-b[0])+abs(a[1]-b[1])\n    def move(a,b):\n        dx,dy=b[0]-a[0],b[1]-a[1]\n        return [\'EAST\' if dx>0 else \'WEST\'] if abs(dx)>=abs(dy) and dx else [\'SOUTH\' if dy>0 else \'NORTH\']\n    coords=[(x,y) for y in range(n) for x in range(n) if tiles[y][x]!=\'LOCKED\']\n    animals=[(x,y) for x,y in coords if isinstance(tiles[y][x],dict) and tiles[y][x].get(\'animal\')]\n    counts={c:sum(isinstance(tiles[y][x],dict) and tiles[y][x].get(\'crop\')==c for x,y in coords) for c in CROPS}\n    nlive=len(animals);claimed=set();actions=[]\n    # Planting estimates account for crops already growing, maturity and expected market saturation.\n    def crop_value(c):\n        cost,first,peak,yld=CROPS[c]\n        if day+first>=29 or (hour>=21 and c!=\'WHEAT\'):return -1\n        lifespan=min(peak,remaining-2)\n        if c in (\'TOMATO\',\'STRAWBERRY\'):\n            interval=1 if c==\'TOMATO\' else 2;yld=max(0,min(4,1+(remaining-2-first)//interval))\n        price=prices[c]\n        if c in (\'MELON\',\'STRAWBERRY\'):price=max(BASE[c]*.20,price-counts[c]*(18 if c==\'MELON\' else 12))\n        if c==\'WHEAT\' and nlive:price=max(price,32)\n        return (yld*price-cost)/(lifespan+4)*CFG[\'crop_bias\']\n    crop_scores={c:crop_value(c) for c in CROPS}\n    for idx,pos0 in enumerate(positions):\n        pos=tuple(pos0);inv=invs[idx];best=(-1,None,None);carried=next((a for a in ANIMALS if inv.get(a,0)),None)\n        nearest_shed=min(access,key=lambda q:dist(pos,q));ds=dist(pos,nearest_shed)\n        def offer(value,target,act,key=None):\n            nonlocal best\n            if key is not None and key in claimed:return\n            distance=dist(pos,target)\n            if distance>23-hour:return\n            score=value/(1+distance*.85)\n            if score>best[0]:best=(score,target,(act,key))\n        # Deposits make produce sellable before night and before the final turn.\n        goods=sum(v for k,v in inv.items() if k not in (\'WHEAT\',\'FERTILIZER\') and k not in ANIMALS)\n        goods_value=sum(v*prices.get(k,0) for k,v in inv.items() if k not in ANIMALS and k!=\'WHEAT\')\n        deposit=(day==29 and hour+ds>=19 and sum(inv.values())>0) or goods>=5 or inv.get(\'FERTILIZER\',0)>=4\n        if deposit and not carried:\n            urgency=3000 if day==29 and hour+ds>=20 else max(90,goods_value*.3)\n            offer(urgency,nearest_shed,[\'DROP\'])\n        if carried:\n            kind=\'COOP\' if carried==\'GOOSE\' else \'PASTURE\'\n            for x,y in coords:\n                t=tiles[y][x]\n                if t is None:offer(280-5*min(dist((x,y),s) for s in access),(x,y),[\'BUILD_COOP\' if kind==\'COOP\' else \'BUILD_PASTURE\'],(\'tile\',x,y))\n                elif isinstance(t,dict) and t.get(\'kind\')==kind and not t.get(\'animal\'):offer(600,(x,y),[\'PLACE\',carried,1],(\'tile\',x,y))\n        else:\n            pending=next((a for a in ANIMALS if shed.get(a,0)>0),None)\n            if pending and day<20:offer(350,nearest_shed,[\'PICKUP\',pending,1])\n        if nlive and inv.get(\'WHEAT\',0)==0 and shed.get(\'WHEAT\',0)>0 and hour<22:\n            offer(210,nearest_shed,[\'PICKUP\',\'WHEAT\',min(4,shed[\'WHEAT\'])])\n        for x,y in coords:\n            t=tiles[y][x];target=(x,y);key=(\'tile\',x,y)\n            if key in claimed:continue\n            if isinstance(t,dict) and t.get(\'animal\'):\n                species=t[\'animal\'];cost,product,first,interval=ANIMALS[species];price=prices[product]\n                if not t.get(\'fed_today\') and inv.get(\'WHEAT\',0):offer(190+130*t.get(\'consecutive_unfed\',0)+hour*3,target,[\'FEED\'],key)\n                if t.get(\'yield_units\',0):offer(60+t[\'yield_units\']*max(price,10)*.8,target,[\'HARVEST\'],key)\n                if not t.get(\'cared_today\') and day<29 and t.get(\'yield_units\',0)<5:offer(max(12,min(price,BASE[product]*1.3))*CFG[\'care_bias\'],target,[\'CARE\'],key)\n                if t.get(\'fertilizer_available\'):offer(max(40,prices[\'FERTILIZER\'])*.85,target,[\'COLLECT_FERTILIZER\'],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'PLANT\':\n                c=t[\'crop\'];cost,first,peak,yld=CROPS[c];age=day-t[\'planted_day\'];yield_units=t.get(\'yield_units\',0)\n                fertile=t.get(\'fertilized_until_day\',-1)>=day\n                needs_fert=(c in (\'TOMATO\',\'STRAWBERRY\') and age>=first-2) or (c not in (\'TOMATO\',\'STRAWBERRY\') and age>=max(0,first-1) and age<peak)\n                if not fertile and needs_fert and inv.get(\'FERTILIZER\',0) and day<29:\n                    offer(min(200,prices[c]*1.4)*CFG[\'fert_bias\'],target,[\'FERTILIZE\'],key)\n                ready=yield_units>0 and age>=first and (c in (\'TOMATO\',\'STRAWBERRY\') or age>=peak or yield_units>=6 or day==29)\n                if ready:offer(100+yield_units*prices[c]*.55+(200 if day==29 else 0),target,[\'HARVEST\'],key)\n                if not t.get(\'watered_today\') and not (day==29 and ready):offer(95+hour*4+120*t.get(\'consecutive_unwatered\',0),target,[\'WATER\'],key)\n            elif t is None and not carried and hour<22:\n                available=[c for c in CROPS if seeds.get(c,0)>0 and crop_scores[c]>0]\n                if available:\n                    c=max(available,key=lambda c:crop_scores[c]);offer(max(12,crop_scores[c]),target,[\'PLANT\',c],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'WEED\' and day<26:offer(20,target,[\'DIG\'],key)\n        if best[1] is None:action=[\'PASS\']\n        else:\n            _,target,(task,key)=best\n            if key:claimed.add(key)\n            if pos!=target:action=move(pos,target)\n            else:\n                action=task;op=action[0];x,y=pos;t=tiles[y][x]\n                # Project same-turn changes to coordinate later units and market orders.\n                if op==\'PLANT\':\n                    seeds[action[1]]-=1;tiles[y][x]={\'kind\':\'PLANT\',\'crop\':action[1],\'planted_day\':day,\'yield_units\':0,\'watered_today\':False}\n                elif op==\'DIG\':tiles[y][x]=None\n                elif op==\'WATER\':t[\'watered_today\']=True\n                elif op==\'FEED\':t[\'fed_today\']=True;inv[\'WHEAT\']-=1\n                elif op==\'CARE\':t[\'cared_today\']=True\n                elif op==\'FERTILIZE\':t[\'fertilized_until_day\']=day+2;inv[\'FERTILIZER\']-=1\n                elif op==\'COLLECT_FERTILIZER\':t[\'fertilizer_available\']=False;inv[\'FERTILIZER\']=inv.get(\'FERTILIZER\',0)+1\n                elif op==\'HARVEST\':\n                    product=ANIMALS[t[\'animal\']][1] if t.get(\'animal\') else t[\'crop\'];inv[product]=inv.get(product,0)+t[\'yield_units\'];t[\'yield_units\']=0\n                    if t.get(\'crop\') in (\'WHEAT\',\'CARROT\',\'MELON\'):tiles[y][x]=None\n                elif op.startswith(\'BUILD_\'):tiles[y][x]={\'kind\':op[6:]}\n                elif op==\'PLACE\' and action[1] in ANIMALS:inv[action[1]]-=1;tiles[y][x]={\'kind\':\'COOP\' if action[1]==\'GOOSE\' else \'PASTURE\',\'animal\':action[1],\'yield_units\':0,\'placed_day\':day}\n                elif op==\'PICKUP\':\n                    k=action[1];amount=min(action[2],shed.get(k,0));shed[k]-=amount;inv[k]=inv.get(k,0)+amount\n                elif op==\'DROP\':\n                    room=max(0,100-sum(shed.values()))\n                    for k,v in list(inv.items()):\n                        take=min(room,v);shed[k]=shed.get(k,0)+take;room-=take;inv[k]=0\n        actions.append(action)\n    orders=[];cash=me[\'money\'];reserve_feed=0 if day==29 else nlive*2\n    for k,q in shed.items():\n        if k in ANIMALS:continue\n        sell=max(0,q-(reserve_feed if k==\'WHEAT\' else 0))\n        if sell:orders.append([\'SELL\',k,sell])\n    # Spend only money already in the bank; each planned purchase reserves its cost.\n    def buy(order,cost):\n        nonlocal cash\n        if cost<=cash and len(orders)<10:orders.append(order);cash-=cost;return True\n        return False\n    target_hands=min(CFG[\'hands\'],4+day//2)\n    if day>=28:target_hands=max(5,CFG[\'hands\']-1)\n    fib=[1,1,2,3,5,8,13,21,34,55,89,144]\n    for h in range(me[\'hires_today\'],target_hands):\n        if hour>12 or cash<fib[h]+40:break\n        buy([\'HIRE\'],fib[h])\n    total_animals=nlive+sum(shed.get(a,0) for a in ANIMALS)+sum(inv.get(a,0) for inv in invs for a in ANIMALS)\n    if day<12 and total_animals<min(CFG[\'animals\'],5+day):\n        populations={a:sum(tiles[y][x].get(\'animal\')==a for x,y in animals)+shed.get(a,0)+sum(v.get(a,0) for v in invs) for a in ANIMALS}\n        values={a:(max(prices[ANIMALS[a][1]],BASE[ANIMALS[a][1]]*.2)*(1+ANIMALS[a][3])/ANIMALS[a][3])/(1+populations[a]*.35) for a in ANIMALS}\n        species=max(values,key=values.get)\n        if cash>ANIMALS[species][0]+200:buy([\'BUY_ANIMAL\',species,1],ANIMALS[species][0])\n    if nlive and hour<22:\n        deficit=max(0,nlive*2-shed.get(\'WHEAT\',0));qty=min(deficit,int(max(0,cash-50)//(prices[\'WHEAT\']+2)))\n        if qty:buy([\'BUY_PRODUCT\',\'WHEAT\',qty],qty*(prices[\'WHEAT\']+2))\n    for c in sorted(CROPS,key=lambda c:-crop_scores[c]):\n        if crop_scores[c]>0 and seeds.get(c,0)<2 and cash>100:\n            qty=2-seeds.get(c,0);buy([\'BUY_SEED\',c,qty],CROPS[c][0]*qty)\n    landcount=len(me[\'unlocked_quadrants\'])\n    if landcount<CFG[\'land\'] and day<18:\n        cost=[1000,2000,4000][landcount-1]\n        if cash>cost+700:buy([\'BUY_LAND\'],cost)\n    return {\'farmer\':actions[0],\'hands\':actions[1:],\'market\':orders[:10]}\n', 'main_v11.py': '"""Kaggriculture: coordinated farming policy. v11: deadline-aware liquidation. Standard library only."""\nimport copy\n\nCFG={\'animals\':12,\'hands\':10,\'land\':3,\'crop_bias\':1.0,\'care_bias\':1.0,\'fert_bias\':1.0}\nCROPS={\'WHEAT\':(10,2,4,4),\'CARROT\':(20,2,3,3),\'MELON\':(80,10,10,6),\'TOMATO\':(50,8,11,4),\'STRAWBERRY\':(100,10,16,4)}\nANIMALS={\'COW\':(400,\'MILK\',8,2),\'SHEEP\':(500,\'WOOL\',6,3),\'GOOSE\':(300,\'EGG\',4,1)}\nBASE={\'WHEAT\':25,\'CARROT\':35,\'MELON\':250,\'TOMATO\':60,\'STRAWBERRY\':120,\'MILK\':160,\'WOOL\':200,\'EGG\':50,\'FERTILIZER\':100}\n\ndef agent(obs):\n    me=copy.deepcopy(obs[\'farms\'][obs[\'player\']]);pr=copy.deepcopy(obs[\'private\'])\n    tiles=me[\'tiles\'];n=len(tiles);day=obs[\'day\'];hour=obs[\'hour\'];prices=obs[\'market\'][\'prices\'];remaining=30-day\n    shed=pr[\'shed\'];seeds=pr[\'seeds\'];invs=pr[\'inventories\'];positions=[me[\'farmer\']]+me[\'hands\']\n    center=n//2;access=[(center-1,center-1),(center,center-1),(center-1,center),(center,center)]\n    def dist(a,b):return abs(a[0]-b[0])+abs(a[1]-b[1])\n    # Movement deliberately permits crossing unowned tiles.\n    def move(a,b):\n        dx,dy=b[0]-a[0],b[1]-a[1]\n        return [\'EAST\' if dx>0 else \'WEST\'] if abs(dx)>=abs(dy) and dx else [\'SOUTH\' if dy>0 else \'NORTH\']\n    coords=[(x,y) for y in range(n) for x in range(n) if tiles[y][x]!=\'LOCKED\']\n    animals=[(x,y) for x,y in coords if isinstance(tiles[y][x],dict) and tiles[y][x].get(\'animal\')]\n    counts={c:sum(isinstance(tiles[y][x],dict) and tiles[y][x].get(\'crop\')==c for x,y in coords) for c in CROPS}\n    nlive=len(animals);claimed=set();actions=[]\n    # Planting estimates account for crops already growing, maturity and expected market saturation.\n    def crop_value(c):\n        cost,first,peak,yld=CROPS[c]\n        if day+first>=29 or (hour>=21 and c!=\'WHEAT\'):return -1\n        lifespan=min(peak,remaining-2)\n        if c in (\'TOMATO\',\'STRAWBERRY\'):\n            interval=1 if c==\'TOMATO\' else 2;yld=max(0,min(4,1+(remaining-2-first)//interval))\n        price=prices[c]\n        if c in (\'MELON\',\'STRAWBERRY\'):price=max(BASE[c]*.20,price-counts[c]*(18 if c==\'MELON\' else 12))\n        if c==\'WHEAT\' and nlive:price=max(price,32)\n        return (yld*price-cost)/(lifespan+4)*CFG[\'crop_bias\']\n    crop_scores={c:crop_value(c) for c in CROPS}\n    for idx,pos0 in enumerate(positions):\n        pos=tuple(pos0);inv=invs[idx];best=(-1,None,None);carried=next((a for a in ANIMALS if inv.get(a,0)),None)\n        nearest_shed=min(access,key=lambda q:dist(pos,q));ds=dist(pos,nearest_shed)\n        def offer(value,target,act,key=None):\n            nonlocal best\n            if key is not None and key in claimed:return\n            distance=dist(pos,target)\n            if day==29:\n                # Last actionable observation is hour 22 in the 720-step game.\n                available=23-hour\n                if act[0] not in (\'HARVEST\',\'COLLECT_FERTILIZER\',\'DROP\'):return\n                delivery=0 if act[0]==\'DROP\' else min(dist(target,s) for s in access)+1\n                if distance+1+delivery>available:return\n            if distance>23-hour:return\n            score=value/(1+distance*.85)\n            if score>best[0]:best=(score,target,(act,key))\n        # Deposits make produce sellable before night and before the final turn.\n        goods=sum(v for k,v in inv.items() if k not in (\'WHEAT\',\'FERTILIZER\') and k not in ANIMALS)\n        goods_value=sum(v*prices.get(k,0) for k,v in inv.items() if k not in ANIMALS and k!=\'WHEAT\')\n        deposit=(day==29 and hour+ds>=19 and sum(inv.values())>0) or goods>=5 or inv.get(\'FERTILIZER\',0)>=4\n        if day==29 and sum(inv.values())>0:deposit=True\n        if deposit and not carried:\n            urgency=3000 if day==29 and hour+ds>=20 else max(90,goods_value*.3)\n            offer(urgency,nearest_shed,[\'DROP\'])\n        if carried:\n            kind=\'COOP\' if carried==\'GOOSE\' else \'PASTURE\'\n            for x,y in coords:\n                t=tiles[y][x]\n                if t is None:offer(280-5*min(dist((x,y),s) for s in access),(x,y),[\'BUILD_COOP\' if kind==\'COOP\' else \'BUILD_PASTURE\'],(\'tile\',x,y))\n                elif isinstance(t,dict) and t.get(\'kind\')==kind and not t.get(\'animal\'):offer(600,(x,y),[\'PLACE\',carried,1],(\'tile\',x,y))\n        else:\n            pending=next((a for a in ANIMALS if shed.get(a,0)>0),None)\n            if pending and day<20:offer(350,nearest_shed,[\'PICKUP\',pending,1])\n        if nlive and inv.get(\'WHEAT\',0)==0 and shed.get(\'WHEAT\',0)>0 and hour<22:\n            offer(210,nearest_shed,[\'PICKUP\',\'WHEAT\',min(4,shed[\'WHEAT\'])])\n        for x,y in coords:\n            t=tiles[y][x];target=(x,y);key=(\'tile\',x,y)\n            if key in claimed:continue\n            if isinstance(t,dict) and t.get(\'animal\'):\n                species=t[\'animal\'];cost,product,first,interval=ANIMALS[species];price=prices[product]\n                if not t.get(\'fed_today\') and inv.get(\'WHEAT\',0):offer(190+130*t.get(\'consecutive_unfed\',0)+hour*3,target,[\'FEED\'],key)\n                if t.get(\'yield_units\',0):offer(60+t[\'yield_units\']*max(price,10)*.8,target,[\'HARVEST\'],key)\n                if not t.get(\'cared_today\') and day<29 and t.get(\'yield_units\',0)<5:offer(max(12,min(price,BASE[product]*1.3))*CFG[\'care_bias\'],target,[\'CARE\'],key)\n                if t.get(\'fertilizer_available\'):offer(max(40,prices[\'FERTILIZER\'])*.85,target,[\'COLLECT_FERTILIZER\'],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'PLANT\':\n                c=t[\'crop\'];cost,first,peak,yld=CROPS[c];age=day-t[\'planted_day\'];yield_units=t.get(\'yield_units\',0)\n                fertile=t.get(\'fertilized_until_day\',-1)>=day\n                needs_fert=(c in (\'TOMATO\',\'STRAWBERRY\') and age>=first-2) or (c not in (\'TOMATO\',\'STRAWBERRY\') and age>=max(0,first-1) and age<peak)\n                if not fertile and needs_fert and inv.get(\'FERTILIZER\',0) and day<29:\n                    offer(min(200,prices[c]*1.4)*CFG[\'fert_bias\'],target,[\'FERTILIZE\'],key)\n                ready=yield_units>0 and age>=first and (c in (\'TOMATO\',\'STRAWBERRY\') or age>=peak or yield_units>=6 or day==29)\n                if ready:offer(100+yield_units*prices[c]*.55+(200 if day==29 else 0),target,[\'HARVEST\'],key)\n                if not t.get(\'watered_today\') and not (day==29 and ready):offer(95+hour*4+120*t.get(\'consecutive_unwatered\',0),target,[\'WATER\'],key)\n            elif t is None and not carried and hour<22:\n                available=[c for c in CROPS if seeds.get(c,0)>0 and crop_scores[c]>0]\n                if available:\n                    c=max(available,key=lambda c:crop_scores[c]);offer(max(12,crop_scores[c]),target,[\'PLANT\',c],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'WEED\' and day<26:offer(20,target,[\'DIG\'],key)\n        if best[1] is None:action=[\'PASS\']\n        else:\n            _,target,(task,key)=best\n            if key:claimed.add(key)\n            if pos!=target:action=move(pos,target)\n            else:\n                action=task;op=action[0];x,y=pos;t=tiles[y][x]\n                # Project same-turn changes to coordinate later units and market orders.\n                if op==\'PLANT\':\n                    seeds[action[1]]-=1;tiles[y][x]={\'kind\':\'PLANT\',\'crop\':action[1],\'planted_day\':day,\'yield_units\':0,\'watered_today\':False}\n                elif op==\'DIG\':tiles[y][x]=None\n                elif op==\'WATER\':t[\'watered_today\']=True\n                elif op==\'FEED\':t[\'fed_today\']=True;inv[\'WHEAT\']-=1\n                elif op==\'CARE\':t[\'cared_today\']=True\n                elif op==\'FERTILIZE\':t[\'fertilized_until_day\']=day+2;inv[\'FERTILIZER\']-=1\n                elif op==\'COLLECT_FERTILIZER\':t[\'fertilizer_available\']=False;inv[\'FERTILIZER\']=inv.get(\'FERTILIZER\',0)+1\n                elif op==\'HARVEST\':\n                    product=ANIMALS[t[\'animal\']][1] if t.get(\'animal\') else t[\'crop\'];inv[product]=inv.get(product,0)+t[\'yield_units\'];t[\'yield_units\']=0\n                    if t.get(\'crop\') in (\'WHEAT\',\'CARROT\',\'MELON\'):tiles[y][x]=None\n                elif op.startswith(\'BUILD_\'):tiles[y][x]={\'kind\':op[6:]}\n                elif op==\'PLACE\' and action[1] in ANIMALS:inv[action[1]]-=1;tiles[y][x]={\'kind\':\'COOP\' if action[1]==\'GOOSE\' else \'PASTURE\',\'animal\':action[1],\'yield_units\':0,\'placed_day\':day}\n                elif op==\'PICKUP\':\n                    k=action[1];amount=min(action[2],shed.get(k,0));shed[k]-=amount;inv[k]=inv.get(k,0)+amount\n                elif op==\'DROP\':\n                    room=max(0,100-sum(shed.values()))\n                    for k,v in list(inv.items()):\n                        take=min(room,v);shed[k]=shed.get(k,0)+take;room-=take;inv[k]=0\n        actions.append(action)\n    orders=[];cash=me[\'money\'];reserve_feed=0 if day==29 else nlive*2\n    for k,q in shed.items():\n        if k in ANIMALS:continue\n        sell=max(0,q-(reserve_feed if k==\'WHEAT\' else 0))\n        if sell:orders.append([\'SELL\',k,sell])\n    # Spend only money already in the bank; each planned purchase reserves its cost.\n    def buy(order,cost):\n        nonlocal cash\n        if cost<=cash and len(orders)<10:orders.append(order);cash-=cost;return True\n        return False\n    target_hands=min(CFG[\'hands\'],4+day//2)\n    if day>=28:target_hands=CFG[\'hands\']\n    fib=[1,1,2,3,5,8,13,21,34,55,89,144]\n    for h in range(me[\'hires_today\'],target_hands):\n        if hour>12 or cash<fib[h]+40:break\n        buy([\'HIRE\'],fib[h])\n    total_animals=nlive+sum(shed.get(a,0) for a in ANIMALS)+sum(inv.get(a,0) for inv in invs for a in ANIMALS)\n    if day<12 and total_animals<min(CFG[\'animals\'],5+day):\n        populations={a:sum(tiles[y][x].get(\'animal\')==a for x,y in animals)+shed.get(a,0)+sum(v.get(a,0) for v in invs) for a in ANIMALS}\n        values={a:(max(prices[ANIMALS[a][1]],BASE[ANIMALS[a][1]]*.2)*(1+ANIMALS[a][3])/ANIMALS[a][3])/(1+populations[a]*.35) for a in ANIMALS}\n        species=max(values,key=values.get)\n        if cash>ANIMALS[species][0]+200:buy([\'BUY_ANIMAL\',species,1],ANIMALS[species][0])\n    if nlive and day<29 and hour<22:\n        deficit=max(0,nlive*2-shed.get(\'WHEAT\',0));qty=min(deficit,int(max(0,cash-50)//(prices[\'WHEAT\']+2)))\n        if qty:buy([\'BUY_PRODUCT\',\'WHEAT\',qty],qty*(prices[\'WHEAT\']+2))\n    for c in sorted(CROPS,key=lambda c:-crop_scores[c]):\n        if crop_scores[c]>0 and seeds.get(c,0)<2 and cash>100:\n            qty=2-seeds.get(c,0);buy([\'BUY_SEED\',c,qty],CROPS[c][0]*qty)\n    landcount=len(me[\'unlocked_quadrants\'])\n    if landcount<CFG[\'land\'] and day<18:\n        cost=[1000,2000,4000][landcount-1]\n        if cash>cost+700:buy([\'BUY_LAND\'],cost)\n    return {\'farmer\':actions[0],\'hands\':actions[1:],\'market\':orders[:10]}\n', 'main_v12.py': '"""Kaggriculture: coordinated farming policy. v12 search template. Standard library only."""\nimport copy\n\nCFG = {\'animals\': 10, \'hands\': 9, \'land\': 2, \'crop_bias\': 1.674270690403267, \'care_bias\': 0.7224837832484434, \'fert_bias\': 1.3793708390542747, \'opponent_weight\': 0.0, \'liquidate\': True, \'drop_units\': 3, \'drop_value\': 1000000, \'cash_release\': True, \'deposit_bias\': 0.5561745205917409, \'feed_fix\': False, \'care_cap\': 1.3, \'plant_floor\': 60, \'hire_pace\': 1, \'workload_hiring\': False, \'work_per_hand\': 6, \'keep_late_hands\': True, \'land_util\': 0.35, \'land_buffer\': 700}\nCROPS={\'WHEAT\':(10,2,4,4),\'CARROT\':(20,2,3,3),\'MELON\':(80,10,10,6),\'TOMATO\':(50,8,11,4),\'STRAWBERRY\':(100,10,16,4)}\nANIMALS={\'COW\':(400,\'MILK\',8,2),\'SHEEP\':(500,\'WOOL\',6,3),\'GOOSE\':(300,\'EGG\',4,1)}\nBASE={\'WHEAT\':25,\'CARROT\':35,\'MELON\':250,\'TOMATO\':60,\'STRAWBERRY\':120,\'MILK\':160,\'WOOL\':200,\'EGG\':50,\'FERTILIZER\':100}\n\ndef agent(obs):\n    me=copy.deepcopy(obs[\'farms\'][obs[\'player\']]);pr=copy.deepcopy(obs[\'private\'])\n    tiles=me[\'tiles\'];n=len(tiles);day=obs[\'day\'];hour=obs[\'hour\'];prices=obs[\'market\'][\'prices\'];remaining=30-day\n    shed=pr[\'shed\'];seeds=pr[\'seeds\'];invs=pr[\'inventories\'];positions=[me[\'farmer\']]+me[\'hands\']\n    center=n//2;access=[(center-1,center-1),(center,center-1),(center-1,center),(center,center)]\n    def dist(a,b):return abs(a[0]-b[0])+abs(a[1]-b[1])\n    def move(a,b):\n        dx,dy=b[0]-a[0],b[1]-a[1]\n        return [\'EAST\' if dx>0 else \'WEST\'] if abs(dx)>=abs(dy) and dx else [\'SOUTH\' if dy>0 else \'NORTH\']\n    coords=[(x,y) for y in range(n) for x in range(n) if tiles[y][x]!=\'LOCKED\']\n    animals=[(x,y) for x,y in coords if isinstance(tiles[y][x],dict) and tiles[y][x].get(\'animal\')]\n    counts={c:sum(isinstance(tiles[y][x],dict) and tiles[y][x].get(\'crop\')==c for x,y in coords) for c in CROPS}\n    nlive=len(animals);claimed=set();actions=[]\n    other=obs[\'farms\'][1-obs[\'player\']]\n    opponent_crops={c:sum(isinstance(t,dict) and t.get(\'crop\')==c for row in other[\'tiles\'] for t in row) for c in CROPS}\n    occupied=sum(isinstance(tiles[y][x],dict) and bool(tiles[y][x].get(\'crop\') or tiles[y][x].get(\'animal\')) for x,y in coords)\n\n    # Planting estimates account for crops already growing, maturity and expected market saturation.\n    def crop_value(c):\n        cost,first,peak,yld=CROPS[c]\n        if day+first>=29 or (hour>=21 and c!=\'WHEAT\'):return -1\n        lifespan=min(peak,remaining-2)\n        if c in (\'TOMATO\',\'STRAWBERRY\'):\n            interval=1 if c==\'TOMATO\' else 2;yld=max(0,min(4,1+(remaining-2-first)//interval))\n        price=prices[c]\n        if CFG[\'opponent_weight\']>0:price=max(BASE[c]*.15,price-CFG[\'opponent_weight\']*opponent_crops[c]*BASE[c]*.06)\n        if c in (\'MELON\',\'STRAWBERRY\'):price=max(BASE[c]*.20,price-counts[c]*(18 if c==\'MELON\' else 12))\n        if c==\'WHEAT\' and nlive:price=max(price,32)\n        return (yld*price-cost)/(lifespan+4)*CFG[\'crop_bias\']\n    crop_scores={c:crop_value(c) for c in CROPS}\n    for idx,pos0 in enumerate(positions):\n        pos=tuple(pos0);inv=invs[idx];best=(-1,None,None);carried=next((a for a in ANIMALS if inv.get(a,0)),None)\n        nearest_shed=min(access,key=lambda q:dist(pos,q));ds=dist(pos,nearest_shed)\n        def offer(value,target,act,key=None):\n            nonlocal best\n            if key is not None and key in claimed:return\n            distance=dist(pos,target)\n            if day==29 and CFG[\'liquidate\']:\n                if act[0] not in (\'HARVEST\',\'COLLECT_FERTILIZER\',\'DROP\'):return\n                delivery=0 if act[0]==\'DROP\' else min(dist(target,s) for s in access)+1\n                if distance+1+delivery>23-hour:return\n            if distance>23-hour:return\n            score=value/(1+distance*.85)\n            if score>best[0]:best=(score,target,(act,key))\n        # Deposits make produce sellable before night and before the final turn.\n        goods=sum(v for k,v in inv.items() if k not in (\'WHEAT\',\'FERTILIZER\') and k not in ANIMALS)\n        goods_value=sum(v*prices.get(k,0) for k,v in inv.items() if k not in ANIMALS and k!=\'WHEAT\')\n        deposit=(day==29 and hour+ds>=19 and sum(inv.values())>0) or goods>=CFG[\'drop_units\'] or inv.get(\'FERTILIZER\',0)>=4\n        deposit=deposit or goods_value>=CFG[\'drop_value\'] or (CFG[\'cash_release\'] and me[\'money\']<100 and goods_value>0)\n        if day==29 and CFG[\'liquidate\'] and sum(inv.values())>0:deposit=True\n        if deposit and not carried:\n            urgency=3000 if day==29 and hour+ds>=20 else max(90,goods_value*CFG[\'deposit_bias\'])\n            offer(urgency,nearest_shed,[\'DROP\'])\n        if carried:\n            kind=\'COOP\' if carried==\'GOOSE\' else \'PASTURE\'\n            for x,y in coords:\n                t=tiles[y][x]\n                if t is None:offer(280-5*min(dist((x,y),s) for s in access),(x,y),[\'BUILD_COOP\' if kind==\'COOP\' else \'BUILD_PASTURE\'],(\'tile\',x,y))\n                elif isinstance(t,dict) and t.get(\'kind\')==kind and not t.get(\'animal\'):offer(600,(x,y),[\'PLACE\',carried,1],(\'tile\',x,y))\n        else:\n            pending=next((a for a in ANIMALS if shed.get(a,0)>0),None)\n            if pending and day<20:offer(350,nearest_shed,[\'PICKUP\',pending,1])\n        if nlive and inv.get(\'WHEAT\',0)==0 and shed.get(\'WHEAT\',0)>0 and hour<22 and (not CFG[\'feed_fix\'] or (day<29 and any(not tiles[y][x].get(\'fed_today\') for x,y in animals))):\n            offer(210,nearest_shed,[\'PICKUP\',\'WHEAT\',min(4,shed[\'WHEAT\'])])\n        for x,y in coords:\n            t=tiles[y][x];target=(x,y);key=(\'tile\',x,y)\n            if key in claimed:continue\n            if isinstance(t,dict) and t.get(\'animal\'):\n                species=t[\'animal\'];cost,product,first,interval=ANIMALS[species];price=prices[product]\n                if not t.get(\'fed_today\') and inv.get(\'WHEAT\',0):offer(190+130*t.get(\'consecutive_unfed\',0)+hour*3,target,[\'FEED\'],key)\n                if t.get(\'yield_units\',0):offer(60+t[\'yield_units\']*max(price,10)*.8,target,[\'HARVEST\'],key)\n                if not t.get(\'cared_today\') and day<29 and t.get(\'yield_units\',0)<5:offer(max(12,min(price,BASE[product]*CFG[\'care_cap\']))*CFG[\'care_bias\'],target,[\'CARE\'],key)\n                if t.get(\'fertilizer_available\'):offer(max(40,prices[\'FERTILIZER\'])*.85,target,[\'COLLECT_FERTILIZER\'],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'PLANT\':\n                c=t[\'crop\'];cost,first,peak,yld=CROPS[c];age=day-t[\'planted_day\'];yield_units=t.get(\'yield_units\',0)\n                fertile=t.get(\'fertilized_until_day\',-1)>=day\n                needs_fert=(c in (\'TOMATO\',\'STRAWBERRY\') and age>=first-2) or (c not in (\'TOMATO\',\'STRAWBERRY\') and age>=max(0,first-1) and age<peak)\n                if not fertile and needs_fert and inv.get(\'FERTILIZER\',0) and day<29:\n                    offer(min(200,prices[c]*1.4)*CFG[\'fert_bias\'],target,[\'FERTILIZE\'],key)\n                ready=yield_units>0 and age>=first and (c in (\'TOMATO\',\'STRAWBERRY\') or age>=peak or yield_units>=6 or day==29)\n                if ready:offer(100+yield_units*prices[c]*.55+(200 if day==29 else 0),target,[\'HARVEST\'],key)\n                if not t.get(\'watered_today\') and not (day==29 and ready):offer(95+hour*4+120*t.get(\'consecutive_unwatered\',0),target,[\'WATER\'],key)\n            elif t is None and not carried and hour<22:\n                available=[c for c in CROPS if seeds.get(c,0)>0 and crop_scores[c]>0]\n                if available:\n                    c=max(available,key=lambda c:crop_scores[c]);offer(max(CFG[\'plant_floor\'],crop_scores[c]),target,[\'PLANT\',c],key)\n            elif isinstance(t,dict) and t.get(\'kind\')==\'WEED\' and day<26:offer(20,target,[\'DIG\'],key)\n        if best[1] is None:action=[\'PASS\']\n        else:\n            _,target,(task,key)=best\n            if key:claimed.add(key)\n            if pos!=target:action=move(pos,target)\n            else:\n                action=task;op=action[0];x,y=pos;t=tiles[y][x]\n                # Project same-turn changes to coordinate later units and market orders.\n                if op==\'PLANT\':\n                    seeds[action[1]]-=1;tiles[y][x]={\'kind\':\'PLANT\',\'crop\':action[1],\'planted_day\':day,\'yield_units\':0,\'watered_today\':False}\n                elif op==\'DIG\':tiles[y][x]=None\n                elif op==\'WATER\':t[\'watered_today\']=True\n                elif op==\'FEED\':t[\'fed_today\']=True;inv[\'WHEAT\']-=1\n                elif op==\'CARE\':t[\'cared_today\']=True\n                elif op==\'FERTILIZE\':t[\'fertilized_until_day\']=day+2;inv[\'FERTILIZER\']-=1\n                elif op==\'COLLECT_FERTILIZER\':t[\'fertilizer_available\']=False;inv[\'FERTILIZER\']=inv.get(\'FERTILIZER\',0)+1\n                elif op==\'HARVEST\':\n                    product=ANIMALS[t[\'animal\']][1] if t.get(\'animal\') else t[\'crop\'];inv[product]=inv.get(product,0)+t[\'yield_units\'];t[\'yield_units\']=0\n                    if t.get(\'crop\') in (\'WHEAT\',\'CARROT\',\'MELON\'):tiles[y][x]=None\n                elif op.startswith(\'BUILD_\'):tiles[y][x]={\'kind\':op[6:]}\n                elif op==\'PLACE\' and action[1] in ANIMALS:inv[action[1]]-=1;tiles[y][x]={\'kind\':\'COOP\' if action[1]==\'GOOSE\' else \'PASTURE\',\'animal\':action[1],\'yield_units\':0,\'placed_day\':day}\n                elif op==\'PICKUP\':\n                    k=action[1];amount=min(action[2],shed.get(k,0));shed[k]-=amount;inv[k]=inv.get(k,0)+amount\n                elif op==\'DROP\':\n                    room=max(0,100-sum(shed.values()))\n                    for k,v in list(inv.items()):\n                        take=min(room,v);shed[k]=shed.get(k,0)+take;room-=take;inv[k]=0\n        actions.append(action)\n    orders=[];cash=me[\'money\'];reserve_feed=0 if day==29 else nlive*2\n    for k,q in shed.items():\n        if k in ANIMALS:continue\n        sell=max(0,q-(reserve_feed if k==\'WHEAT\' else 0))\n        if sell:orders.append([\'SELL\',k,sell])\n    # Spend only money already in the bank; each planned purchase reserves its cost.\n    def buy(order,cost):\n        nonlocal cash\n        if cost<=cash and len(orders)<10:orders.append(order);cash-=cost;return True\n        return False\n    target_hands=min(CFG[\'hands\'],4+day//CFG[\'hire_pace\'])\n    if CFG[\'workload_hiring\']:\n        workload=sum(counts.values())+nlive*3\n        target_hands=min(CFG[\'hands\'],max(4,2+int(workload/CFG[\'work_per_hand\'])))\n    if day>=28:target_hands=CFG[\'hands\'] if CFG[\'keep_late_hands\'] else max(5,CFG[\'hands\']-1)\n    fib=[1,1,2,3,5,8,13,21,34,55,89,144]\n    for h in range(me[\'hires_today\'],target_hands):\n        if hour>12 or cash<fib[h]+40:break\n        buy([\'HIRE\'],fib[h])\n    total_animals=nlive+sum(shed.get(a,0) for a in ANIMALS)+sum(inv.get(a,0) for inv in invs for a in ANIMALS)\n    if day<12 and total_animals<min(CFG[\'animals\'],5+day):\n        populations={a:sum(tiles[y][x].get(\'animal\')==a for x,y in animals)+shed.get(a,0)+sum(v.get(a,0) for v in invs) for a in ANIMALS}\n        values={a:(max(prices[ANIMALS[a][1]],BASE[ANIMALS[a][1]]*.2)*(1+ANIMALS[a][3])/ANIMALS[a][3])/(1+populations[a]*.35) for a in ANIMALS}\n        species=max(values,key=values.get)\n        if cash>ANIMALS[species][0]+200:buy([\'BUY_ANIMAL\',species,1],ANIMALS[species][0])\n    if nlive and hour<22 and (day<29 or not CFG[\'liquidate\']):\n        deficit=max(0,nlive*2-shed.get(\'WHEAT\',0));qty=min(deficit,int(max(0,cash-50)//(prices[\'WHEAT\']+2)))\n        if qty:buy([\'BUY_PRODUCT\',\'WHEAT\',qty],qty*(prices[\'WHEAT\']+2))\n    for c in sorted(CROPS,key=lambda c:-crop_scores[c]):\n        if crop_scores[c]>0 and seeds.get(c,0)<2 and cash>100:\n            qty=2-seeds.get(c,0);buy([\'BUY_SEED\',c,qty],CROPS[c][0]*qty)\n    landcount=len(me[\'unlocked_quadrants\'])\n    if landcount<CFG[\'land\'] and day<18 and occupied/max(1,len(coords))>=CFG[\'land_util\']:\n        cost=[1000,2000,4000][landcount-1]\n        if cash>cost+CFG[\'land_buffer\']:buy([\'BUY_LAND\'],cost)\n    return {\'farmer\':actions[0],\'hands\':actions[1:],\'market\':orders[:10]}\n'}
for name, source in EMBEDDED_FILES.items():
    compile(source,name,'exec')
    (CODE/name).write_text(source)
print('Embedded v2 league and policies ready.')


## Throughput benchmark (chooses the worker count)

Matches are independent CPU processes, so throughput scales with worker count up to
the number of cores. This cell times `BENCH_GAMES` games at a few candidate counts
and sets `WORKERS` to the fastest. Set `AUTO_BENCHMARK_WORKERS = False` to skip it.


In [ ]:
import json, time, textwrap
if AUTO_BENCHMARK_WORKERS:
    bench = textwrap.dedent('''
        import sys, time, contextlib, io, importlib.util, concurrent.futures, multiprocessing, json
        from pathlib import Path
        CODE=Path(sys.argv[1]); candidates=[int(x) for x in sys.argv[2].split(",")]; games=int(sys.argv[3])
        def _play(args):
            seed,=args
            with contextlib.redirect_stdout(io.StringIO()),contextlib.redirect_stderr(io.StringIO()):
                from kaggle_environments import make
            def load(p,n):
                s=importlib.util.spec_from_file_location(n,str(p)); m=importlib.util.module_from_spec(s); s.loader.exec_module(m); return m.agent
            A=load(CODE/"main_v12.py","A"); B=load(CODE/"main_v9.py","B")
            e=make("kaggriculture",configuration={"seed":seed,"episodeSteps":720}); e.run([A,B]); return 1
        if __name__=="__main__":
            ctx=multiprocessing.get_context("spawn"); jobs=[(40000000+i,) for i in range(games)]; results={}
            for w in candidates:
                t0=time.perf_counter()
                with concurrent.futures.ProcessPoolExecutor(max_workers=w,mp_context=ctx) as pool:
                    list(pool.map(_play,jobs))
                dt=time.perf_counter()-t0; results[w]=games/dt
                print(f"  workers={w:>2}  {dt:6.1f}s  {games/dt:5.2f} games/s",flush=True)
            best=max(results,key=results.get); print("BEST_WORKERS="+str(best),flush=True)
        ''')
    (CODE/'bench.py').write_text(bench)
    cpus=os.cpu_count() or 2
    cand=sorted({1,2,4,cpus} & set(range(1,cpus+1))) or [1]
    print(f'Benchmarking worker counts {cand} on {cpus} logical CPUs ({BENCH_GAMES} games each)...')
    proc=subprocess.run([str(PYTHON),str(CODE/'bench.py'),str(CODE),','.join(map(str,cand)),str(BENCH_GAMES)],
                        capture_output=True,text=True)
    print(proc.stdout.strip())
    best=[l for l in proc.stdout.splitlines() if l.startswith('BEST_WORKERS=')]
    if best:
        WORKERS=int(best[0].split('=')[1]); print('Selected WORKERS =',WORKERS)
    else:
        WORKERS=max(1,min(4,cpus)); print('Benchmark inconclusive; WORKERS =',WORKERS)
        print(proc.stderr[-800:])
else:
    print('Auto-benchmark disabled; WORKERS =',WORKERS)

## Restore and persistence

For Kaggle, preserve the saved notebook outputs and attach the checkpoint to a later session. For Colab, optionally mount Drive and set CHECKPOINT_COPY above, or download the archive before the runtime expires. Saving to a runtime's local disk alone does not guarantee survival after that runtime disappears.

Checkpoints are written at trial reservations/completions and phase boundaries. A graceful time limit also saves partial game caches. A hard runtime kill can lose work since the last copied checkpoint. On resume, a pending trial uses its original parameters; a reserved holdout uses the same candidate and seed block. A candidate is not repeatedly retested on new holdouts until it happens to pass.

Keep code, opponents, trial/seed settings and evaluation targets unchanged when resuming. Runtime, worker count, MAX_CYCLES, STAGNATION_CYCLES, STOP_ON_TARGET and checkpoint-copy location may change. Use a fresh output folder to start a different experiment. Optimizer randomness after resume need not match an uninterrupted session exactly.


In [ ]:
import json, zipfile, shutil
if RESUME_ZIP:
    if (OUTPUT/'state.json').exists() or (OUTPUT/'study.db').exists():
        raise RuntimeError('Output already contains a study. Clear RESUME_ZIP to continue it.')
    allowed={'state.json','manifest.json','main.py','games.jsonl','decision.json','study.db'}
    folders={'agents','candidates','reports','replays','sources'}
    with zipfile.ZipFile(RESUME_ZIP) as z:
        for m in z.infolist():
            rel=Path(m.filename);target=(OUTPUT/rel).resolve()
            if OUTPUT.resolve() not in target.parents:
                raise ValueError('Unsafe archive path')
            if str(rel) not in allowed and rel.parts[0] not in folders:
                raise ValueError('Unexpected archive entry: '+m.filename)
        z.extractall(OUTPUT)

config=dict(output_dir=str(OUTPUT),time_budget_minutes=TIME_BUDGET_MINUTES,
            max_cycles=MAX_CYCLES,general_trials=GENERAL_TRIALS,exploiter_trials=EXPLOITER_TRIALS,
            workers=WORKERS,train_seeds=TRAIN_SEEDS,selection_seeds=SELECTION_SEEDS,
            holdout_seeds=HOLDOUT_SEEDS,finalists=FINALISTS,selfplay_seeds=SELFPLAY_SEEDS,
            history_size=HISTORY_SIZE,target_mean_cash=TARGET_MEAN_CASH,target_win_rate=TARGET_WIN_RATE,
            target_80k_rate=TARGET_80K_RATE,stop_on_target=STOP_ON_TARGET,
            stagnation_cycles=STAGNATION_CYCLES,v9_win_rate=V9_WIN_RATE,
            champion_win_rate=CHAMPION_WIN_RATE,min_opponent_win_rate=MIN_OPPONENT_WIN_RATE,
            max_decision_seconds=MAX_DECISION_SECONDS,extra_opponents=EXTRA_OPPONENTS,
            checkpoint_copy=CHECKPOINT_COPY)
(CODE/'config.json').write_text(json.dumps(config,indent=2))
print('Ready to start or continue the league.')


## Run the evolving league

Each cycle: self-play diagnostics → generalist search → targeted exploiter search → common-league selection → unseen holdout → promotion/counterstrategy archive → next cycle. Target achievement triggers a second independent confirmation before stopping.

The league and champion remain frozen during a cycle so comparisons have a stable opponent set. Training, selection, holdout, confirmation and counterstrategy seeds are disjoint and renewed each cycle. Both player positions are used. Fresh seeds test game variation; they do not introduce unseen opponent algorithms by themselves. Local execution speed is not a guarantee of hosted runtime speed.

An eight-hour budget does not promise a fixed number of cycles. Match logs show actual progress; a partial cycle resumes where it stopped. Pure self-play ties or a higher maximum cash value do not trigger promotion.


In [ ]:
environment=dict(os.environ,OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1')
command=[str(PYTHON),str(CODE/'league.py'),'--config',str(CODE/'config.json')]
with (OUTPUT/'training.log').open('a') as log:
    process=subprocess.Popen(command,cwd=str(CODE),env=environment,stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        return_code=process.wait()
    except KeyboardInterrupt:
        import signal
        process.send_signal(signal.SIGINT)
        print('Stop requested. Allow the trainer to save its checkpoint before closing the runtime.')
        raise
if return_code:
    raise RuntimeError('Inspect training.log. Preserve checkpoint.zip before restarting.')


## Results

`promoted: false` means the export is still the embedded V12 fallback. Do not spend a submission on the unchanged file. A counterstrategy being added is not a promotion. `target_reached: true` means the local target passed the additional confirmation, not that Kaggle rank or rating is guaranteed.

Reports contain per-opponent outcomes, seed-cluster uncertainty, cash, 80k hit rate/timing, land use, crops, animals, staffing, route changes and action counts. Farm snapshot day keys are zero-based. Four-quadrant ownership and productive use are recorded separately. Counts of actions issued do not necessarily equal successful actions. All game cash includes starting money.


In [ ]:
decision=json.loads((OUTPUT/'decision.json').read_text())
print(json.dumps(decision,indent=2))
print('Champion/history:')
state=json.loads((OUTPUT/'state.json').read_text())
print('Completed cycles:',state['cycle'],'| Promotions:',state['promotions'],'| Archived agents:',len(state['archives']))
if decision.get('latest_report'):
    report=json.loads((OUTPUT/decision['latest_report']).read_text())
    print(json.dumps(report['summary'],indent=2))
shutil.copyfile(OUTPUT/'main.py',WORKING/'main.py')
shutil.copyfile(OUTPUT/'checkpoint.zip',WORKING/'enhanced_game_v1_checkpoint.zip')
from IPython.display import display,FileLink
display(FileLink(str(WORKING/'main.py')))
display(FileLink(str(WORKING/'enhanced_game_v1_checkpoint.zip')))
print('No Kaggle submission was made.')


In [ ]:
# Optional Colab downloads:
# from google.colab import files
# files.download(str(WORKING/'main.py'))
# files.download(str(WORKING/'enhanced_game_v1_checkpoint.zip'))


## Method and limits

This is Optuna TPE search over an explicit Python policy, not neural reinforcement
learning and not GPU computation. A profile of one game showed the time goes to
Python object copying (`deepcopy`, roughly half), struct conversion and per-step
schema validation inside the CPU simulator; there is no dense numeric work a GPU
could accelerate without rewriting the simulator. The honest throughput lever is CPU
process parallelism, which the benchmark cell now sets automatically.

The phase-dependent parameters and the widened `plant_floor` enlarge the search
space; they do not by themselves make a stronger agent. Whether a distinct late-game
policy wins is decided only by the fresh-holdout promotion gates against V9, the
champion and the league. `crop_style`, `animal_style`, per-region routing and the
mid-game (`*_mid`) phase keys remain available as further levers.

Checkpoints are Enhanced Game v2 format and are not interchangeable with v1, V12 or
V13 archives. No Kaggle submission is made and no leaderboard improvement is
promised; report measured results from the run, not assumed ones.
